<a href="https://colab.research.google.com/github/vinodm-kapdaa/Colab_for_training/blob/main/Macro_router_from_March.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
============================================================
   HELD-OUT EVALUATION — All 4 Macro Router Models
   Test on Careys labelled data (never seen during training)
============================================================
Usage (Colab):
    !pip install --upgrade transformers>=4.56.0 timm>=1.0.20 --quiet
    Then run this script.
"""

import os
import cv2
import json
import time
import numpy as np
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score
)
from tqdm import tqdm

# ==============================================================
# CONFIG
# ==============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Careys held-out data (NEVER used in training)
EVAL_ROOT = "/content/drive/Shareddrives/Garment Type/classified_images_13_14_test"

# Trained model checkpoints (searched in order)
MODEL_DIRS = [
    "/content/drive/Shareddrives/Garment Type/model_comparison",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison",
]

# Where to save evaluation results
OUTPUT_DIR = "/content/drive/Shareddrives/Garment Type/model_comparison/eval_results"

BATCH_SIZE = 32
NUM_WORKERS = 4

# --- Preprocessing (must match training exactly) ---
APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING (must match training)
# ==============================================================

UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece",
                 "blazer", "jumpers", "shirt", "t-shirt", "dresses", "fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts",
                 "jeans", "trousers", "jogging_bottoms", "skirts"]
OTHER_CLASSES = ["Towel", "towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker", "bra", "kincker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# ==============================================================
# MODEL CONFIGS (must match training)
# ==============================================================

MODEL_CONFIGS = {
    "SigLIP2_SO400M": {
        "model_id": "google/siglip2-so400m-patch14-384",
        "type": "siglip2",
        "img_size": 384,
        "unfreeze_layers": 6,
    },
    "DINOv3_ViT_Base": {
        "model_id": "vit_base_patch16_dinov3.lvd1689m",
        "type": "timm",
        "img_size": 384,
        "unfreeze_layers": 6,
    },
    "ConvNeXt_DINOv3": {
        "model_id": "convnext_base.dinov3_lvd1689m",
        "type": "timm",
        "img_size": 384,
        "unfreeze_layers": 2,
    },
    "EfficientNetV2_S": {
        "model_id": "tf_efficientnetv2_s",
        "type": "timm",
        "img_size": 384,
        "unfreeze_layers": 2,
    },
}

# ==============================================================
# PREPROCESSING (exact copy from training)
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped

    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))


# ==============================================================
# DATASET
# ==============================================================

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None


def collect_samples(root):
    """Collect (path, label, original_class) tuples from folder structure."""
    samples = []
    if not os.path.isdir(root):
        print(f"  ❌ Directory not found: {root}")
        return samples
    for cls_name in sorted(os.listdir(root)):
        cls_path = os.path.join(root, cls_name)
        if not os.path.isdir(cls_path):
            continue
        label = map_class(cls_name)
        if label is None:
            print(f"  ⚠️  Skipping unmapped class: {cls_name}")
            continue
        count = 0
        for f in os.listdir(cls_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(cls_path, f), label, cls_name))
                count += 1
        print(f"  {cls_name:20s} → {IDX_TO_CLASS[label]:10s} ({count} images)")
    return samples


class EvalDataset(Dataset):
    def __init__(self, samples, transform, normalize_fn=None, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.normalize_fn = normalize_fn
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, orig_cls = self.samples[idx]

        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (384, 384), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")

        pixel_values = self.transform(img)
        if self.normalize_fn:
            pixel_values = self.normalize_fn(pixel_values)

        return pixel_values, label, idx  # return idx to trace back to sample


# ==============================================================
# CLASSIFIER HEAD (must match training)
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.fc(self.dropout(x))


# ==============================================================
# MODEL LOADING
# ==============================================================

def load_backbone(config):
    """Load vision backbone architecture (no weights yet)."""
    model_type = config["type"]
    model_id = config["model_id"]

    if model_type == "siglip2":
        from transformers import AutoProcessor
        try:
            from transformers import Siglip2VisionModel
            vision_model = Siglip2VisionModel.from_pretrained(model_id)
        except Exception:
            try:
                from transformers import SiglipVisionModel
                vision_model = SiglipVisionModel.from_pretrained(model_id)
            except Exception:
                from transformers import AutoModel
                full = AutoModel.from_pretrained(model_id)
                vision_model = full.vision_model
                del full

        hidden_size = vision_model.config.hidden_size
        normalize_fn = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

        def extract_fn(model, pixel_values):
            return model(pixel_values=pixel_values).pooler_output

        return vision_model, hidden_size, normalize_fn, extract_fn

    elif model_type == "timm":
        import timm
        vision_model = timm.create_model(model_id, pretrained=False, num_classes=0)
        hidden_size = vision_model.num_features
        data_cfg = timm.data.resolve_data_config(model=vision_model)
        normalize_fn = transforms.Normalize(mean=data_cfg["mean"], std=data_cfg["std"])

        def extract_fn(model, pixel_values):
            return model(pixel_values)

        return vision_model, hidden_size, normalize_fn, extract_fn


def find_checkpoint(model_name):
    """Search for checkpoint across all model directories."""
    for d in MODEL_DIRS:
        path = os.path.join(d, f"{model_name}_best.pt")
        if os.path.exists(path):
            return path
    return None


def load_checkpoint(model_name, config):
    """Load trained checkpoint and return (vision_model, classifier, normalize_fn, extract_fn, meta)."""
    ckpt_path = find_checkpoint(model_name)
    if ckpt_path is None:
        raise FileNotFoundError(f"Checkpoint not found for {model_name} in: {MODEL_DIRS}")

    # Load architecture
    vision_model, hidden_size, normalize_fn, extract_fn = load_backbone(config)

    # Load checkpoint
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    # Override hidden_size if checkpoint has it
    if "hidden_size" in ckpt:
        hidden_size = ckpt["hidden_size"]

    # Load weights
    vision_model.load_state_dict(ckpt["vision_model"])
    classifier = ClassifierHead(hidden_size, NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])

    meta = {
        "best_f1_train": ckpt.get("best_f1", None),
        "best_epoch": ckpt.get("epoch", None),
    }

    return vision_model, classifier, normalize_fn, extract_fn, meta


# ==============================================================
# EVALUATION
# ==============================================================

@torch.no_grad()
def evaluate_model(vision_model, classifier, extract_fn, dataloader):
    """Run inference and return predictions, ground truths, confidences, per-sample details."""
    vision_model.eval()
    classifier.eval()

    all_preds = []
    all_gts = []
    all_confs = []
    all_probs = []
    all_indices = []

    for batch_pixels, batch_labels, batch_idx in tqdm(dataloader, desc="  Evaluating"):
        batch_pixels = batch_pixels.to(DEVICE)

        features = extract_fn(vision_model, batch_pixels)
        logits = classifier(features)
        probs = torch.softmax(logits, dim=1)
        confs, preds = probs.max(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_gts.extend(batch_labels.numpy())
        all_confs.extend(confs.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_indices.extend(batch_idx.numpy())

    return (
        np.array(all_preds),
        np.array(all_gts),
        np.array(all_confs),
        np.array(all_probs),
        np.array(all_indices),
    )


# ==============================================================
# ANALYSIS HELPERS
# ==============================================================

def compute_metrics(preds, gts, class_names):
    """Compute comprehensive metrics."""
    f1_macro = f1_score(gts, preds, average="macro")
    f1_weighted = f1_score(gts, preds, average="weighted")
    accuracy = accuracy_score(gts, preds)
    precision = precision_score(gts, preds, average="macro", zero_division=0)
    recall = recall_score(gts, preds, average="macro", zero_division=0)

    per_class_f1 = f1_score(gts, preds, average=None, zero_division=0)
    cm = confusion_matrix(gts, preds)
    report = classification_report(gts, preds, target_names=class_names, zero_division=0)

    return {
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "per_class_f1": {class_names[i]: float(per_class_f1[i]) for i in range(len(class_names))},
        "confusion_matrix": cm.tolist(),
        "report": report,
    }


def find_misclassified(preds, gts, confs, indices, samples):
    """Find all misclassified samples with details."""
    misclassified = []
    for i in range(len(preds)):
        if preds[i] != gts[i]:
            sample_idx = indices[i]
            path, label, orig_cls = samples[sample_idx]
            misclassified.append({
                "file": os.path.basename(path),
                "folder": orig_cls,
                "full_path": path,
                "true_label": IDX_TO_CLASS[gts[i]],
                "pred_label": IDX_TO_CLASS[preds[i]],
                "confidence": float(confs[i]),
            })
    # Sort by confidence descending (high-confidence mistakes are worst)
    misclassified.sort(key=lambda x: x["confidence"], reverse=True)
    return misclassified


def find_uncertain(preds, gts, confs, indices, samples, threshold=0.60):
    """Find samples below confidence threshold (would be 'Unknown' in production)."""
    uncertain = []
    for i in range(len(preds)):
        if confs[i] < threshold:
            sample_idx = indices[i]
            path, label, orig_cls = samples[sample_idx]
            uncertain.append({
                "file": os.path.basename(path),
                "folder": orig_cls,
                "true_label": IDX_TO_CLASS[gts[i]],
                "pred_label": IDX_TO_CLASS[preds[i]],
                "confidence": float(confs[i]),
                "would_be_correct": preds[i] == gts[i],
            })
    uncertain.sort(key=lambda x: x["confidence"])
    return uncertain


def print_confusion_matrix(cm, class_names, model_name):
    """Pretty print confusion matrix."""
    header = f"{'':>12s}" + "".join(f"{c:>10s}" for c in class_names)
    print(f"\n  {model_name} Confusion Matrix:")
    print(f"  {header}")
    for i, row in enumerate(cm):
        row_str = f"  {class_names[i]:>12s}" + "".join(f"{v:>10d}" for v in row)
        print(row_str)


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 70)
    print("   HELD-OUT EVALUATION — Careys Labelled Data")
    print("   (Never seen during training)")
    print("=" * 70)
    print(f"Device: {DEVICE}")
    print(f"Eval root: {EVAL_ROOT}")
    print(f"Model dirs: {MODEL_DIRS}\n")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ----------------------------------------------------------
    # 1. Collect evaluation samples
    # ----------------------------------------------------------
    print("Collecting evaluation samples...")
    samples = collect_samples(EVAL_ROOT)
    print(f"\nTotal evaluation samples: {len(samples)}")

    if len(samples) == 0:
        print("❌ No samples found. Check EVAL_ROOT path.")
        return

    # Show distribution
    dist = defaultdict(int)
    sub_dist = defaultdict(int)
    for _, label, orig_cls in samples:
        dist[IDX_TO_CLASS[label]] += 1
        sub_dist[orig_cls] += 1

    print("\nMacro class distribution:")
    for cls_name in CLASS_TO_IDX:
        print(f"  {cls_name:12s}: {dist[cls_name]:5d}")

    print("\nSub-class distribution:")
    for cls_name, count in sorted(sub_dist.items()):
        print(f"  {cls_name:20s}: {count:5d}")

    # ----------------------------------------------------------
    # 2. Evaluate each model
    # ----------------------------------------------------------
    class_names = list(CLASS_TO_IDX.keys())
    all_model_results = {}

    for model_name, config in MODEL_CONFIGS.items():
        ckpt_path = find_checkpoint(model_name)
        if ckpt_path is None:
            print(f"\n⚠️  SKIPPING {model_name} — checkpoint not found in: {MODEL_DIRS}")
            continue

        print(f"\n{'='*70}")
        print(f"  EVALUATING: {model_name}")
        print(f"  Checkpoint: {ckpt_path}")
        print(f"{'='*70}")

        # Load model
        t_load_start = time.perf_counter()
        try:
            vision_model, classifier, normalize_fn, extract_fn, meta = \
                load_checkpoint(model_name, config)
        except Exception as e:
            print(f"  ❌ Failed to load model: {e}")
            continue

        vision_model = vision_model.to(DEVICE)
        classifier = classifier.to(DEVICE)
        t_load = time.perf_counter() - t_load_start
        print(f"  Model loaded in {t_load:.1f}s (train F1={meta['best_f1_train']}, epoch={meta['best_epoch']})")

        # Build dataloader
        val_transform = transforms.Compose([
            transforms.Resize((config["img_size"], config["img_size"])),
            transforms.ToTensor(),
        ])
        dataset = EvalDataset(samples, val_transform, normalize_fn, APPLY_PREPROCESSING)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS,
                                pin_memory=True)

        # Run inference
        t_infer_start = time.perf_counter()
        preds, gts, confs, probs, indices = evaluate_model(
            vision_model, classifier, extract_fn, dataloader
        )
        t_infer = time.perf_counter() - t_infer_start
        fps = len(samples) / t_infer

        # Compute metrics
        metrics = compute_metrics(preds, gts, class_names)

        # Find misclassified + uncertain
        misclassified = find_misclassified(preds, gts, confs, indices, samples)
        uncertain_60 = find_uncertain(preds, gts, confs, indices, samples, threshold=0.60)
        uncertain_70 = find_uncertain(preds, gts, confs, indices, samples, threshold=0.70)

        # Confidence stats
        correct_mask = preds == gts
        mean_conf_correct = float(confs[correct_mask].mean()) if correct_mask.sum() > 0 else 0
        mean_conf_wrong = float(confs[~correct_mask].mean()) if (~correct_mask).sum() > 0 else 0

        # Per-class confidence
        per_class_conf = {}
        for cls_idx, cls_name in IDX_TO_CLASS.items():
            cls_mask = gts == cls_idx
            if cls_mask.sum() > 0:
                per_class_conf[cls_name] = {
                    "mean_confidence": float(confs[cls_mask].mean()),
                    "min_confidence": float(confs[cls_mask].min()),
                    "correct_pct": float(correct_mask[cls_mask].mean() * 100),
                }

        # Print results
        print(f"\n  📊 Results for {model_name}:")
        print(f"  {'─'*50}")
        print(f"  F1 (macro):     {metrics['f1_macro']:.4f}")
        print(f"  F1 (weighted):  {metrics['f1_weighted']:.4f}")
        print(f"  Accuracy:       {metrics['accuracy']:.4f}")
        print(f"  Precision:      {metrics['precision_macro']:.4f}")
        print(f"  Recall:         {metrics['recall_macro']:.4f}")
        print(f"  Inference:      {t_infer:.1f}s ({fps:.1f} img/s)")
        print(f"  {'─'*50}")
        print(f"  Per-class F1:")
        for cls_name in class_names:
            f1_val = metrics["per_class_f1"][cls_name]
            conf_info = per_class_conf.get(cls_name, {})
            mean_c = conf_info.get("mean_confidence", 0)
            min_c = conf_info.get("min_confidence", 0)
            pct = conf_info.get("correct_pct", 0)
            print(f"    {cls_name:12s}: F1={f1_val:.4f}  "
                  f"conf={mean_c:.3f} (min={min_c:.3f})  "
                  f"acc={pct:.1f}%")

        print_confusion_matrix(metrics["confusion_matrix"], class_names, model_name)

        print(f"\n  Confidence analysis:")
        print(f"    Mean confidence (correct):  {mean_conf_correct:.4f}")
        print(f"    Mean confidence (wrong):    {mean_conf_wrong:.4f}")
        print(f"    Samples < 0.60 threshold:   {len(uncertain_60)} "
              f"({len(uncertain_60)/len(samples)*100:.1f}%)")
        print(f"    Samples < 0.70 threshold:   {len(uncertain_70)} "
              f"({len(uncertain_70)/len(samples)*100:.1f}%)")
        print(f"    Total misclassified:        {len(misclassified)} "
              f"({len(misclassified)/len(samples)*100:.1f}%)")

        # Show worst mistakes (high confidence + wrong)
        if misclassified:
            print(f"\n  🔴 Top 15 worst mistakes (high confidence, wrong prediction):")
            for m in misclassified[:15]:
                print(f"    {m['file']:40s} | {m['folder']:15s} → "
                      f"true={m['true_label']:10s} pred={m['pred_label']:10s} "
                      f"conf={m['confidence']:.3f}")

        # Store results
        all_model_results[model_name] = {
            "metrics": metrics,
            "meta": meta,
            "inference_time_s": float(t_infer),
            "fps": float(fps),
            "confidence": {
                "mean_correct": float(mean_conf_correct),
                "mean_wrong": float(mean_conf_wrong),
                "per_class": per_class_conf,
            },
            "misclassified_count": len(misclassified),
            "misclassified_top20": misclassified[:20],
            "uncertain_below_060": len(uncertain_60),
            "uncertain_below_070": len(uncertain_70),
        }

        # Cleanup
        del vision_model, classifier
        torch.cuda.empty_cache()
        import gc; gc.collect()

    # ----------------------------------------------------------
    # 3. Final comparison table
    # ----------------------------------------------------------
    if len(all_model_results) == 0:
        print("\n❌ No models evaluated.")
        return

    print(f"\n\n{'='*90}")
    print(f"   FINAL HELD-OUT COMPARISON — Careys Data")
    print(f"{'='*90}")

    header = (f"{'Model':24s} {'F1':>6s} {'Acc':>6s} {'Prec':>6s} {'Rec':>6s} "
              f"{'Upper':>7s} {'Lower':>7s} {'Undwr':>7s} {'Other':>7s} "
              f"{'Errors':>7s} {'FPS':>6s}")
    print(header)
    print("─" * 90)

    best_model = None
    best_f1 = -1

    for model_name, result in all_model_results.items():
        m = result["metrics"]
        pcf = m["per_class_f1"]
        f1 = m["f1_macro"]

        row = (f"{model_name:24s} {f1:>6.4f} {m['accuracy']:>6.4f} "
               f"{m['precision_macro']:>6.4f} {m['recall_macro']:>6.4f} "
               f"{pcf.get('Upper',0):>7.4f} {pcf.get('Lower',0):>7.4f} "
               f"{pcf.get('Underwear',0):>7.4f} {pcf.get('Other',0):>7.4f} "
               f"{result['misclassified_count']:>7d} {result['fps']:>6.1f}")
        print(row)

        if f1 > best_f1:
            best_f1 = f1
            best_model = model_name

    print("─" * 90)
    print(f"\n>>> WINNER on held-out data: {best_model} with F1={best_f1:.4f}")

    # Compare train F1 vs held-out F1 (generalization gap)
    print(f"\n  Generalization gap (train_val F1 → held-out F1):")
    for model_name, result in all_model_results.items():
        train_f1 = result["meta"]["best_f1_train"]
        held_f1 = result["metrics"]["f1_macro"]
        gap = (train_f1 - held_f1) if train_f1 else None
        if gap is not None:
            indicator = "🟢" if gap < 0.03 else ("🟡" if gap < 0.06 else "🔴")
            print(f"    {indicator} {model_name:24s}: {train_f1:.4f} → {held_f1:.4f}  "
                  f"(gap={gap:+.4f})")
        else:
            print(f"    ⚪ {model_name:24s}: ???  → {held_f1:.4f}")

    # ----------------------------------------------------------
    # 4. Save results
    # ----------------------------------------------------------
    results_path = os.path.join(OUTPUT_DIR, "eval_careys_results.json")
    save_data = {
        "eval_root": EVAL_ROOT,
        "total_samples": len(samples),
        "class_distribution": dict(dist),
        "subclass_distribution": dict(sub_dist),
        "winner": best_model,
        "models": {},
    }

    for model_name, result in all_model_results.items():
        # Remove the full classification report (not JSON serializable in a nice way)
        r = result.copy()
        r["metrics"] = {k: v for k, v in r["metrics"].items() if k != "report"}
        save_data["models"][model_name] = r

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(results_path, "w") as f:
        json.dump(save_data, f, indent=2)
    print(f"\n✅ Results saved to: {results_path}")

    # Also save misclassified details for analysis
    misc_path = os.path.join(OUTPUT_DIR, "misclassified_details.json")
    misc_data = {}
    for model_name, result in all_model_results.items():
        misc_data[model_name] = result.get("misclassified_top20", [])
    with open(misc_path, "w") as f:
        json.dump(misc_data, f, indent=2)
    print(f"✅ Misclassified details saved to: {misc_path}")


if __name__ == "__main__":
    main()

   HELD-OUT EVALUATION — Careys Labelled Data
   (Never seen during training)
Device: cuda
Eval root: /content/drive/Shareddrives/Garment Type/classified_images_13_14_test
Model dirs: ['/content/drive/Shareddrives/Garment Type/model_comparison', '/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison']

  ⚠️  Skipping unmapped class: 13_14th_test
  blazer               → Upper      (89 images)
  bra                  → Underwear  (48 images)
  dresses              → Upper      (12 images)
  fleece               → Upper      (63 images)
  jeans                → Lower      (60 images)
  jogging_bottoms      → Lower      (249 images)
  jumpers              → Upper      (181 images)
  kincker              → Underwear  (33 images)
  shirt                → Upper      (165 images)
  skirts               → Lower      (38 images)
  t-shirt              → Upper      (111 images)
  towel                → Other      (31 images)
  trousers             → Lower      (245 images)
  

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

You are using a model of type siglip_vision_model to instantiate a model of type siglip2_vision_model. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

...


  📊 Results for DINOv3_ViT_Base:
  ──────────────────────────────────────────────────
  F1 (macro):     0.8753
  F1 (weighted):  0.9243
  Accuracy:       0.9230
  Precision:      0.8388
  Recall:         0.9294
  Inference:      35.3s (37.5 img/s)
  ──────────────────────────────────────────────────
  Per-class F1:
    Upper       : F1=0.9282  conf=0.511 (min=0.286)  acc=93.7%
    Lower       : F1=0.9311  conf=0.556 (min=0.270)  acc=90.2%
    Underwear   : F1=0.9240  conf=0.930 (min=0.424)  acc=97.5%
    Other       : F1=0.7179  conf=0.865 (min=0.385)  acc=90.3%

  DINOv3_ViT_Base Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       582        21         7        11
         Lower        47       534         3         8
     Underwear         2         0        79         0
         Other         2         0         1        28

  Confidence analysis:
    Mean confidence (correct):  0.5728
    Mean confidence (wrong):    0.4717
    Samples < 0.

  Evaluating: 100%|██████████| 42/42 [00:37<00:00,  1.12it/s]



  📊 Results for ConvNeXt_DINOv3:
  ──────────────────────────────────────────────────
  F1 (macro):     0.8651
  F1 (weighted):  0.9182
  Accuracy:       0.9170
  Precision:      0.8369
  Recall:         0.9058
  Inference:      37.6s (35.2 img/s)
  ──────────────────────────────────────────────────
  Per-class F1:
    Upper       : F1=0.9261  conf=0.513 (min=0.289)  acc=93.9%
    Lower       : F1=0.9181  conf=0.554 (min=0.289)  acc=89.0%
    Underwear   : F1=0.9581  conf=0.933 (min=0.437)  acc=98.8%
    Other       : F1=0.6579  conf=0.840 (min=0.329)  acc=80.6%

  ConvNeXt_DINOv3 Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       583        27         3         8
         Lower        52       527         1        12
     Underwear         1         0        80         0
         Other         2         2         2        25

  Confidence analysis:
    Mean confidence (correct):  0.5727
    Mean confidence (wrong):    0.4763
    Samples < 0.

  Evaluating: 100%|██████████| 42/42 [00:35<00:00,  1.19it/s]



  📊 Results for EfficientNetV2_S:
  ──────────────────────────────────────────────────
  F1 (macro):     0.7034
  F1 (weighted):  0.8391
  Accuracy:       0.8257
  Precision:      0.6748
  Recall:         0.7841
  Inference:      35.4s (37.5 img/s)
  ──────────────────────────────────────────────────
  Per-class F1:
    Upper       : F1=0.8478  conf=0.466 (min=0.281)  acc=83.9%
    Lower       : F1=0.8610  conf=0.529 (min=0.275)  acc=81.1%
    Underwear   : F1=0.8261  conf=0.835 (min=0.394)  acc=93.8%
    Other       : F1=0.2787  conf=0.508 (min=0.323)  acc=54.8%

  EfficientNetV2_S Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       521        42        17        41
         Lower        71       480         8        33
     Underwear         5         0        76         0
         Other        11         1         2        17

  Confidence analysis:
    Mean confidence (correct):  0.5398
    Mean confidence (wrong):    0.4124
    Samples < 

### Train 3 indvidually

In [ ]:
"""
============================================================
   STANDALONE TRAINING — SigLIP2_SO400M
   Garment Macro Router (Upper/Lower/Underwear/Other)
============================================================
Fixes applied vs previous runs:
  1. LABEL_SMOOTHING = 0.0 (was 0.1 — caused 85%+ predictions below 0.60)
  2. Checkpoint verification after save (prevents silent corruption)
  3. Local-first save then copy to Drive (prevents Drive write crash)
"""

import os
import cv2
import numpy as np
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import math
import json
import time
import gc
import shutil

# ==============================================================
# CONFIG
# ==============================================================

MODEL_NAME = "SigLIP2_SO400M"
MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
BATCH_SIZE = 16
LR_HEAD = 2e-4
LR_BACKBONE = 5e-6
UNFREEZE_LAYERS = 6

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data"
]

SAVE_DIR = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models"
LOCAL_SAVE = "/content"  # Fast local SSD for checkpoint writes

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 60
EARLY_STOP_PATIENCE = 12
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.3
LABEL_SMOOTHING = 0.0      # FIX: was 0.1, caused low confidence in production
NUM_WORKERS = 4

APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING
# ==============================================================
UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# ==============================================================
# PREPROCESSING
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

# ==============================================================
# AUGMENTATIONS
# ==============================================================

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.3),
    transforms.RandomRotation(degrees=30, fill=128),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10, fill=128),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3, fill=128),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), value=0.5),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

normalize_fn = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

# ==============================================================
# DATASET
# ==============================================================

class RouterDataset(Dataset):
    def __init__(self, samples, transform, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")
        pixel_values = self.transform(img)
        pixel_values = normalize_fn(pixel_values)
        return pixel_values, label

# ==============================================================
# DATA COLLECTION
# ==============================================================

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

def collect_samples(roots):
    samples = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cls_name in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            label = map_class(cls_name)
            if label is None:
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    samples.append((os.path.join(cls_path, f), label))
    return samples

# ==============================================================
# MODEL
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))

class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        super().__init__(optimizer, last_epoch=-1)
    def get_lr(self):
        step = self.last_epoch
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        return [base_lr * scale for base_lr in self.base_lrs]

def load_model():
    """Load SigLIP2 vision backbone."""
    try:
        from transformers import Siglip2VisionModel
        vision_model = Siglip2VisionModel.from_pretrained(MODEL_ID)
    except Exception:
        try:
            from transformers import SiglipVisionModel
            vision_model = SiglipVisionModel.from_pretrained(MODEL_ID)
        except Exception:
            from transformers import AutoModel
            full = AutoModel.from_pretrained(MODEL_ID)
            vision_model = full.vision_model
            del full

    hidden_size = vision_model.config.hidden_size

    # Freeze all, then unfreeze last N encoder layers + post_layernorm
    for p in vision_model.parameters():
        p.requires_grad = False
    for layer in vision_model.vision_model.encoder.layers[-UNFREEZE_LAYERS:]:
        for p in layer.parameters():
            p.requires_grad = True
    for p in vision_model.vision_model.post_layernorm.parameters():
        p.requires_grad = True

    return vision_model, hidden_size

def extract_features(model, pixel_values):
    return model(pixel_values=pixel_values).pooler_output


# ==============================================================
# PER-CLASS THRESHOLD COMPUTATION
# ==============================================================

def compute_class_thresholds(preds, gts, confs, target_precision=0.95):
    """
    For each class, find the minimum confidence threshold where
    precision >= target_precision. This gives per-class rejection
    thresholds for production use.

    Returns dict: {class_name: threshold}
    """
    thresholds = {}
    preds = np.array(preds)
    gts = np.array(gts)
    confs = np.array(confs)

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        # All samples where model predicted this class
        pred_mask = preds == cls_idx
        if pred_mask.sum() == 0:
            thresholds[cls_name] = 0.99  # No predictions → very high threshold
            continue

        cls_confs = confs[pred_mask]
        cls_correct = (gts[pred_mask] == cls_idx)

        # Sort by confidence descending
        sort_idx = np.argsort(-cls_confs)
        cls_confs_sorted = cls_confs[sort_idx]
        cls_correct_sorted = cls_correct[sort_idx]

        # Walk from highest confidence down, find where precision drops below target
        cumulative_correct = np.cumsum(cls_correct_sorted)
        cumulative_total = np.arange(1, len(cls_correct_sorted) + 1)
        precisions = cumulative_correct / cumulative_total

        # Find the lowest confidence where precision >= target
        valid = np.where(precisions >= target_precision)[0]
        if len(valid) > 0:
            last_valid = valid[-1]
            thresholds[cls_name] = float(cls_confs_sorted[last_valid])
        else:
            # Can't reach target precision — use threshold where precision is maximized
            best_idx = np.argmax(precisions)
            thresholds[cls_name] = float(cls_confs_sorted[best_idx])

        # Also compute stats for logging
        correct_confs = cls_confs[cls_correct]
        wrong_confs = cls_confs[~cls_correct]

        n_total = int(pred_mask.sum())
        n_correct = int(cls_correct.sum())
        prec = n_correct / n_total if n_total > 0 else 0

        print(f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} conf_wrong={wrong_confs.mean():.3f}" if len(wrong_confs) > 0
              else f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} (no wrong predictions)")

    return thresholds


# ==============================================================
# CRASH-SAFE CHECKPOINTING (RESUMABLE)
# --------------------------------------------------------------
# - Writes "<MODEL>_last.pt" every epoch with FULL training state
# - Atomic: tmp file -> os.replace -> copy to Drive
# - Rotates 2 slots on Drive so a crash mid-copy never leaves you empty
# - On startup: auto-loads _last.pt and resumes from next epoch
# ==============================================================

def _atomic_torch_save(obj, final_path):
    """Write to <path>.tmp then atomically rename to <path>."""
    tmp_path = final_path + ".tmp"
    torch.save(obj, tmp_path)
    os.replace(tmp_path, final_path)

def save_last_checkpoint(state, filename, drive_dir):
    """Save resumable training state. Local -> verify -> Drive (rotating)."""
    local_path = os.path.join(LOCAL_SAVE, filename)
    _atomic_torch_save(state, local_path)

    # Verify locally
    try:
        v = torch.load(local_path, map_location="cpu", weights_only=False)
        assert v["epoch"] == state["epoch"]
        del v
    except Exception as e:
        print(f"  LOCAL LAST CKPT VERIFY FAILED: {e}")
        return False

    # Rotate on Drive: current _last.pt -> _last_prev.pt, then copy new
    drive_path = os.path.join(drive_dir, filename)
    drive_prev = os.path.join(drive_dir, filename.replace(".pt", "_prev.pt"))
    try:
        if os.path.exists(drive_path):
            # rotate: move old drive copy to _prev (best-effort)
            try:
                shutil.copy2(drive_path, drive_prev)
            except Exception:
                pass
        # copy new checkpoint to drive
        tmp_drive = drive_path + ".tmp"
        shutil.copy2(local_path, tmp_drive)
        os.replace(tmp_drive, drive_path)
    except Exception as e:
        print(f"  Drive copy of last-ckpt failed (local copy still safe): {e}")
        return False
    return True

def try_load_last_checkpoint(filename, drive_dir):
    """Look for a resumable checkpoint. Tries local, drive, drive_prev in order."""
    candidates = [
        os.path.join(LOCAL_SAVE, filename),
        os.path.join(drive_dir, filename),
        os.path.join(drive_dir, filename.replace(".pt", "_prev.pt")),
    ]
    for p in candidates:
        if not os.path.exists(p):
            continue
        try:
            ckpt = torch.load(p, map_location="cpu", weights_only=False)
            # sanity check
            _ = ckpt["epoch"]; _ = ckpt["vision_model"]; _ = ckpt["optimizer"]
            print(f"  Resume checkpoint found at: {p}")
            print(f"     -> resuming from epoch {ckpt['epoch']+1}, best_f1={ckpt.get('best_f1',0):.4f}")
            return ckpt
        except Exception as e:
            print(f"  Corrupt/incomplete ckpt at {p}: {e}")
            continue
    print("  No resume checkpoint found - starting fresh.")
    return None

# ==============================================================
# CHECKPOINT SAVE + VERIFY
# ==============================================================

def save_checkpoint(vision_model, classifier, hidden_size, best_f1, best_epoch,
                    filename, drive_path):
    """Save locally, verify, then copy to Drive."""
    local_path = os.path.join(LOCAL_SAVE, filename)
    ckpt = {
        "vision_model": vision_model.state_dict(),
        "classifier": classifier.state_dict(),
        "class_to_idx": CLASS_TO_IDX,
        "hidden_size": hidden_size,
        "best_f1": best_f1,
        "epoch": best_epoch,
        "model_name": MODEL_NAME,
        "label_smoothing": LABEL_SMOOTHING,
    }

    # Save locally
    torch.save(ckpt, local_path)

    # Verify: reload and check F1 matches
    verify = torch.load(local_path, map_location="cpu", weights_only=False)
    if abs(verify["best_f1"] - best_f1) > 1e-6:
        print(f"  ❌ CHECKPOINT VERIFICATION FAILED! Saved F1={verify['best_f1']}, expected={best_f1}")
        return False
    print(f"  ✅ Checkpoint verified: F1={verify['best_f1']:.4f}, epoch={verify['epoch']}")
    del verify

    # Copy to Drive
    try:
        shutil.copy2(local_path, drive_path)
        print(f"  ✅ Copied to Drive: {drive_path}")
    except Exception as e:
        print(f"  ⚠️ Drive copy failed (local copy still safe): {e}")

    return True

# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print(f"   TRAINING: {MODEL_NAME}")
    print(f"   Model: {MODEL_ID}")
    print(f"   Label smoothing: {LABEL_SMOOTHING} (fixed — was 0.1)")
    print("=" * 60)
    print(f"Device: {DEVICE}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    # Collect data
    print("\nCollecting samples...")
    all_samples = collect_samples(TRAIN_ROOTS)
    print(f"Total: {len(all_samples)}")

    labels = [l for _, l in all_samples]
    train_samples, val_samples = train_test_split(
        all_samples, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

    # Class weights
    train_labels = [l for _, l in train_samples]
    train_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    print("\nClass distribution:")
    for i in range(NUM_CLASSES):
        print(f"  {IDX_TO_CLASS[i]}: {train_counts[i]}")

    class_weights = torch.tensor(
        1.0 / (train_counts + 1e-6), dtype=torch.float32, device=DEVICE
    )
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES

    # Load model
    vision_model, hidden_size = load_model()
    vision_model = vision_model.to(DEVICE)

    trainable = sum(p.numel() for p in vision_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in vision_model.parameters())
    print(f"\nParams: {total:,}, Trainable: {trainable:,} ({100*trainable/total:.1f}%)")

    classifier = ClassifierHead(hidden_size, NUM_CLASSES).to(DEVICE)

    # Data loaders
    train_loader = DataLoader(
        RouterDataset(train_samples, train_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, drop_last=True,
    )
    val_loader = DataLoader(
        RouterDataset(val_samples, val_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True,
    )

    # Optimizer
    backbone_params = [p for p in vision_model.parameters() if p.requires_grad]
    head_params = list(classifier.parameters())
    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": LR_HEAD},
        {"params": backbone_params, "lr": LR_BACKBONE},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = len(train_loader) * 3
    scheduler = CosineWarmupScheduler(optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda")

    # Training
    best_f1 = 0.0
    best_epoch = 0
    patience = 0
    epoch_logs = []
    best_preds, best_gts, best_confs = [], [], []
    start_epoch = 0

    model_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_best.pt")
    last_ckpt_name = f"{MODEL_NAME}_last.pt"
    start_time = time.time()

    # --- RESUME IF POSSIBLE ---
    print("\n[resume] checking for existing training checkpoint...")
    resume = try_load_last_checkpoint(last_ckpt_name, SAVE_DIR)
    if resume is not None:
        vision_model.load_state_dict(resume["vision_model"])
        classifier.load_state_dict(resume["classifier"])
        optimizer.load_state_dict(resume["optimizer"])
        scheduler.load_state_dict(resume["scheduler"])
        scaler.load_state_dict(resume["scaler"])
        start_epoch     = resume["epoch"] + 1
        best_f1         = resume.get("best_f1", 0.0)
        best_epoch      = resume.get("best_epoch", 0)
        patience        = resume.get("patience", 0)
        epoch_logs      = resume.get("epoch_logs", [])
        best_preds      = resume.get("best_preds", [])
        best_gts        = resume.get("best_gts", [])
        best_confs      = resume.get("best_confs", [])
        # RNG state for reproducibility across resumes
        if "rng_torch" in resume:
            torch.set_rng_state(resume["rng_torch"])
        if "rng_cuda" in resume and torch.cuda.is_available():
            try: torch.cuda.set_rng_state_all(resume["rng_cuda"])
            except Exception: pass
        if "rng_numpy" in resume:
            np.random.set_state(resume["rng_numpy"])
        print(f"[resume] starting at epoch {start_epoch+1}/{EPOCHS} (best F1 so far: {best_f1:.4f})")
        del resume
        gc.collect(); torch.cuda.empty_cache()

    for epoch in range(start_epoch, EPOCHS):
        # Train
        vision_model.train()
        classifier.train()
        total_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"  E{epoch+1}/{EPOCHS} [T]", leave=False)
        for px, y in train_bar:
            px = px.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(backbone_params + head_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)

        # Validate
        vision_model.eval()
        classifier.eval()
        preds, gts, confs = [], [], []
        val_loss = 0.0

        with torch.no_grad(), torch.amp.autocast("cuda"):
            for px, y in val_loader:
                px = px.to(DEVICE, non_blocking=True)
                y_dev = y.to(DEVICE, non_blocking=True)
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                val_loss += criterion(logits, y_dev).item()
                probs = torch.softmax(logits, dim=1)
                max_confs, max_preds = probs.max(dim=1)
                preds.extend(max_preds.cpu().numpy())
                confs_batch = max_confs.cpu().numpy()
                confs.extend(confs_batch)
                gts.extend(y.numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(gts, preds, average="macro")

        epoch_logs.append({
            "epoch": epoch + 1,
            "train_loss": round(avg_loss, 5),
            "val_loss": round(avg_val_loss, 5),
            "val_f1": round(val_f1, 4),
        })

        print(f"  E{epoch+1:03d} | Loss: {avg_loss:.4f}/{avg_val_loss:.4f} | F1: {val_f1:.4f}", end="")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            patience = 0
            best_preds = preds.copy()
            best_gts = gts.copy()
            best_confs = confs.copy()

            ok = save_checkpoint(vision_model, classifier, hidden_size,
                                 best_f1, best_epoch,
                                 f"{MODEL_NAME}_best.pt", model_path)
            print(f" >>> BEST {'✅' if ok else '❌'}")
        else:
            patience += 1
            print(f" ({patience}/{EARLY_STOP_PATIENCE})")
            if patience >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        # --- Save resumable "last" checkpoint every epoch ---
        last_state = {
            "epoch": epoch,
            "vision_model": vision_model.state_dict(),
            "classifier": classifier.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "patience": patience,
            "epoch_logs": epoch_logs,
            "best_preds": best_preds,
            "best_gts": best_gts,
            "best_confs": best_confs,
            "hidden_size": hidden_size,
            "class_to_idx": CLASS_TO_IDX,
            "model_name": MODEL_NAME,
            "label_smoothing": LABEL_SMOOTHING,
            "rng_torch": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "rng_numpy": np.random.get_state(),
        }
        save_last_checkpoint(last_state, last_ckpt_name, SAVE_DIR)
        del last_state

    elapsed = time.time() - start_time

    # Compute per-class thresholds
    class_thresholds = {}
    if best_preds and best_confs:
        print("\nComputing per-class confidence thresholds (target precision=95%):")
        class_thresholds = compute_class_thresholds(best_preds, best_gts, best_confs)
        print(f"\n  Thresholds: {class_thresholds}")

        # Re-save checkpoint with thresholds included
        print("\n  Re-saving checkpoint with per-class thresholds...")
        local_path = os.path.join(LOCAL_SAVE, f"{MODEL_NAME}_best.pt")
        ckpt = torch.load(local_path, map_location="cpu", weights_only=False)
        ckpt["class_thresholds"] = class_thresholds
        torch.save(ckpt, local_path)
        try:
            shutil.copy2(local_path, model_path)
            print(f"  ✅ Checkpoint updated with thresholds")
        except Exception as e:
            print(f"  ⚠️ Drive copy failed: {e}")

    # Final report
    if best_preds and best_gts:
        report = classification_report(best_gts, best_preds,
                                       target_names=list(CLASS_TO_IDX.keys()),
                                       output_dict=True)
        cm = confusion_matrix(best_gts, best_preds)
    else:
        report = {cls: {"f1-score": 0} for cls in CLASS_TO_IDX}
        cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

    print(f"\n{'='*60}")
    print(f"  {MODEL_NAME} DONE")
    print(f"  Best F1: {best_f1:.4f} @ epoch {best_epoch}")
    print(f"  Time: {elapsed/60:.1f} min")
    print(f"{'='*60}")
    print("\nPer-class F1:")
    for cls in CLASS_TO_IDX:
        print(f"  {cls:12s}: {report[cls]['f1-score']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  {'':>12}", end="")
    for name in CLASS_TO_IDX: print(f"{name:>10}", end="")
    print()
    for i, name in enumerate(CLASS_TO_IDX):
        print(f"  {name:>12}", end="")
        for j in range(NUM_CLASSES): print(f"{cm[i][j]:>10}", end="")
        print()

    # Verify final checkpoint one more time
    print(f"\n🔍 Final checkpoint verification:")
    final_ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    print(f"  Checkpoint F1: {final_ckpt['best_f1']:.4f}, Epoch: {final_ckpt['epoch']}")
    assert abs(final_ckpt["best_f1"] - best_f1) < 1e-6, "CHECKPOINT MISMATCH!"
    print(f"  ✅ Verified — checkpoint is correct")

    # Save training log
    log_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_log.json")
    with open(log_path, "w") as f:
        json.dump({
            "model": MODEL_NAME,
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "total_epochs": epoch + 1,
            "train_time_min": round(elapsed / 60, 1),
            "label_smoothing": LABEL_SMOOTHING,
            "per_class_f1": {cls: round(report[cls]["f1-score"], 4) for cls in CLASS_TO_IDX},
            "confusion_matrix": cm.tolist(),
            "class_thresholds": class_thresholds,
            "epoch_logs": epoch_logs,
        }, f, indent=2)
    print(f"  Training log saved: {log_path}")

if __name__ == "__main__":
    main()

In [ ]:
"""
============================================================
   STANDALONE TRAINING — DINOv3_ViT_Base
   Garment Macro Router (Upper/Lower/Underwear/Other)
============================================================
Fixes applied vs previous runs:
  1. LABEL_SMOOTHING = 0.0 (was 0.1 — caused 85%+ predictions below 0.60)
  2. Checkpoint verification after save (prevents silent corruption)
  3. Local-first save then copy to Drive (prevents Drive write crash)
"""

import os
import cv2
import numpy as np
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import math
import json
import time
import gc
import shutil
import timm

# ==============================================================
# CONFIG
# ==============================================================

MODEL_NAME = "DINOv3_ViT_Base"
MODEL_ID = "vit_base_patch16_dinov3.lvd1689m"
IMG_SIZE = 384
BATCH_SIZE = 32
LR_HEAD = 3e-4
LR_BACKBONE = 1e-5
UNFREEZE_LAYERS = 6

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data"
]

SAVE_DIR = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models"
LOCAL_SAVE = "/content"  # Fast local SSD for checkpoint writes


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 60
EARLY_STOP_PATIENCE = 12
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.3
LABEL_SMOOTHING = 0.0      # FIX: was 0.1
NUM_WORKERS = 4

APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING
# ==============================================================
UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# ==============================================================
# PREPROCESSING
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

# ==============================================================
# AUGMENTATIONS + NORMALIZATION
# ==============================================================

# Get timm-specific normalization
_tmp_model = timm.create_model(MODEL_ID, pretrained=False, num_classes=0)
_data_cfg = timm.data.resolve_data_config(model=_tmp_model)
normalize_fn = transforms.Normalize(mean=_data_cfg["mean"], std=_data_cfg["std"])
del _tmp_model

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.3),
    transforms.RandomRotation(degrees=30, fill=128),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10, fill=128),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3, fill=128),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), value=0.5),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ==============================================================
# DATASET
# ==============================================================

class RouterDataset(Dataset):
    def __init__(self, samples, transform, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")
        pixel_values = self.transform(img)
        pixel_values = normalize_fn(pixel_values)
        return pixel_values, label

# ==============================================================
# DATA COLLECTION
# ==============================================================

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

def collect_samples(roots):
    samples = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cls_name in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            label = map_class(cls_name)
            if label is None:
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    samples.append((os.path.join(cls_path, f), label))
    return samples

# ==============================================================
# MODEL
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))

class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        super().__init__(optimizer, last_epoch=-1)
    def get_lr(self):
        step = self.last_epoch
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        return [base_lr * scale for base_lr in self.base_lrs]

def load_model():
    """Load DINOv3 ViT-Base backbone."""
    vision_model = timm.create_model(MODEL_ID, pretrained=True, num_classes=0)
    hidden_size = vision_model.num_features

    # Freeze all
    for p in vision_model.parameters():
        p.requires_grad = False

    # Unfreeze last N blocks (ViT)
    unfrozen = 0
    print(f"  [unfreeze] ViT: {len(vision_model.blocks)} blocks, unfreezing last {UNFREEZE_LAYERS}")
    for block in vision_model.blocks[-UNFREEZE_LAYERS:]:
        for p in block.parameters():
            p.requires_grad = True
            unfrozen += p.numel()
    if hasattr(vision_model, "norm"):
        for p in vision_model.norm.parameters():
            p.requires_grad = True
            unfrozen += p.numel()
    if hasattr(vision_model, "fc_norm"):
        for p in vision_model.fc_norm.parameters():
            p.requires_grad = True
            unfrozen += p.numel()
    print(f"  [unfreeze] Total unfrozen: {unfrozen:,}")

    return vision_model, hidden_size

def extract_features(model, pixel_values):
    return model(pixel_values)


# ==============================================================
# PER-CLASS THRESHOLD COMPUTATION
# ==============================================================

def compute_class_thresholds(preds, gts, confs, target_precision=0.95):
    """
    For each class, find the minimum confidence threshold where
    precision >= target_precision. This gives per-class rejection
    thresholds for production use.

    Returns dict: {class_name: threshold}
    """
    thresholds = {}
    preds = np.array(preds)
    gts = np.array(gts)
    confs = np.array(confs)

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        # All samples where model predicted this class
        pred_mask = preds == cls_idx
        if pred_mask.sum() == 0:
            thresholds[cls_name] = 0.99  # No predictions → very high threshold
            continue

        cls_confs = confs[pred_mask]
        cls_correct = (gts[pred_mask] == cls_idx)

        # Sort by confidence descending
        sort_idx = np.argsort(-cls_confs)
        cls_confs_sorted = cls_confs[sort_idx]
        cls_correct_sorted = cls_correct[sort_idx]

        # Walk from highest confidence down, find where precision drops below target
        cumulative_correct = np.cumsum(cls_correct_sorted)
        cumulative_total = np.arange(1, len(cls_correct_sorted) + 1)
        precisions = cumulative_correct / cumulative_total

        # Find the lowest confidence where precision >= target
        valid = np.where(precisions >= target_precision)[0]
        if len(valid) > 0:
            last_valid = valid[-1]
            thresholds[cls_name] = float(cls_confs_sorted[last_valid])
        else:
            # Can't reach target precision — use threshold where precision is maximized
            best_idx = np.argmax(precisions)
            thresholds[cls_name] = float(cls_confs_sorted[best_idx])

        # Also compute stats for logging
        correct_confs = cls_confs[cls_correct]
        wrong_confs = cls_confs[~cls_correct]

        n_total = int(pred_mask.sum())
        n_correct = int(cls_correct.sum())
        prec = n_correct / n_total if n_total > 0 else 0

        print(f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} conf_wrong={wrong_confs.mean():.3f}" if len(wrong_confs) > 0
              else f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} (no wrong predictions)")

    return thresholds


# ==============================================================
# CRASH-SAFE CHECKPOINTING (RESUMABLE)
# --------------------------------------------------------------
# - Writes "<MODEL>_last.pt" every epoch with FULL training state
# - Atomic: tmp file -> os.replace -> copy to Drive
# - Rotates 2 slots on Drive so a crash mid-copy never leaves you empty
# - On startup: auto-loads _last.pt and resumes from next epoch
# ==============================================================

def _atomic_torch_save(obj, final_path):
    """Write to <path>.tmp then atomically rename to <path>."""
    tmp_path = final_path + ".tmp"
    torch.save(obj, tmp_path)
    os.replace(tmp_path, final_path)

def save_last_checkpoint(state, filename, drive_dir):
    """Save resumable training state. Local -> verify -> Drive (rotating)."""
    local_path = os.path.join(LOCAL_SAVE, filename)
    _atomic_torch_save(state, local_path)

    # Verify locally
    try:
        v = torch.load(local_path, map_location="cpu", weights_only=False)
        assert v["epoch"] == state["epoch"]
        del v
    except Exception as e:
        print(f"  LOCAL LAST CKPT VERIFY FAILED: {e}")
        return False

    # Rotate on Drive: current _last.pt -> _last_prev.pt, then copy new
    drive_path = os.path.join(drive_dir, filename)
    drive_prev = os.path.join(drive_dir, filename.replace(".pt", "_prev.pt"))
    try:
        if os.path.exists(drive_path):
            # rotate: move old drive copy to _prev (best-effort)
            try:
                shutil.copy2(drive_path, drive_prev)
            except Exception:
                pass
        # copy new checkpoint to drive
        tmp_drive = drive_path + ".tmp"
        shutil.copy2(local_path, tmp_drive)
        os.replace(tmp_drive, drive_path)
    except Exception as e:
        print(f"  Drive copy of last-ckpt failed (local copy still safe): {e}")
        return False
    return True

def try_load_last_checkpoint(filename, drive_dir):
    """Look for a resumable checkpoint. Tries local, drive, drive_prev in order."""
    candidates = [
        os.path.join(LOCAL_SAVE, filename),
        os.path.join(drive_dir, filename),
        os.path.join(drive_dir, filename.replace(".pt", "_prev.pt")),
    ]
    for p in candidates:
        if not os.path.exists(p):
            continue
        try:
            ckpt = torch.load(p, map_location="cpu", weights_only=False)
            # sanity check
            _ = ckpt["epoch"]; _ = ckpt["vision_model"]; _ = ckpt["optimizer"]
            print(f"  Resume checkpoint found at: {p}")
            print(f"     -> resuming from epoch {ckpt['epoch']+1}, best_f1={ckpt.get('best_f1',0):.4f}")
            return ckpt
        except Exception as e:
            print(f"  Corrupt/incomplete ckpt at {p}: {e}")
            continue
    print("  No resume checkpoint found - starting fresh.")
    return None

# ==============================================================
# CHECKPOINT SAVE + VERIFY
# ==============================================================

def save_checkpoint(vision_model, classifier, hidden_size, best_f1, best_epoch,
                    filename, drive_path):
    local_path = os.path.join(LOCAL_SAVE, filename)
    ckpt = {
        "vision_model": vision_model.state_dict(),
        "classifier": classifier.state_dict(),
        "class_to_idx": CLASS_TO_IDX,
        "hidden_size": hidden_size,
        "best_f1": best_f1,
        "epoch": best_epoch,
        "model_name": MODEL_NAME,
        "label_smoothing": LABEL_SMOOTHING,
    }
    torch.save(ckpt, local_path)

    verify = torch.load(local_path, map_location="cpu", weights_only=False)
    if abs(verify["best_f1"] - best_f1) > 1e-6:
        print(f"  ❌ CHECKPOINT VERIFICATION FAILED!")
        return False
    print(f"  ✅ Checkpoint verified: F1={verify['best_f1']:.4f}, epoch={verify['epoch']}")
    del verify

    try:
        shutil.copy2(local_path, drive_path)
        print(f"  ✅ Copied to Drive: {drive_path}")
    except Exception as e:
        print(f"  ⚠️ Drive copy failed (local copy safe): {e}")
    return True

# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print(f"   TRAINING: {MODEL_NAME}")
    print(f"   Model: {MODEL_ID}")
    print(f"   Label smoothing: {LABEL_SMOOTHING} (fixed — was 0.1)")
    print("=" * 60)
    print(f"Device: {DEVICE}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    print("\nCollecting samples...")
    all_samples = collect_samples(TRAIN_ROOTS)
    print(f"Total: {len(all_samples)}")

    labels = [l for _, l in all_samples]
    train_samples, val_samples = train_test_split(
        all_samples, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

    train_labels = [l for _, l in train_samples]
    train_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    print("\nClass distribution:")
    for i in range(NUM_CLASSES):
        print(f"  {IDX_TO_CLASS[i]}: {train_counts[i]}")

    class_weights = torch.tensor(
        1.0 / (train_counts + 1e-6), dtype=torch.float32, device=DEVICE
    )
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES

    vision_model, hidden_size = load_model()
    vision_model = vision_model.to(DEVICE)

    trainable = sum(p.numel() for p in vision_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in vision_model.parameters())
    print(f"\nParams: {total:,}, Trainable: {trainable:,} ({100*trainable/total:.1f}%)")

    classifier = ClassifierHead(hidden_size, NUM_CLASSES).to(DEVICE)

    train_loader = DataLoader(
        RouterDataset(train_samples, train_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, drop_last=True,
    )
    val_loader = DataLoader(
        RouterDataset(val_samples, val_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True,
    )

    backbone_params = [p for p in vision_model.parameters() if p.requires_grad]
    head_params = list(classifier.parameters())
    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": LR_HEAD},
        {"params": backbone_params, "lr": LR_BACKBONE},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = len(train_loader) * 3
    scheduler = CosineWarmupScheduler(optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda")

    best_f1 = 0.0
    best_epoch = 0
    patience = 0
    epoch_logs = []
    best_preds, best_gts, best_confs = [], [], []
    start_epoch = 0

    model_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_best.pt")
    last_ckpt_name = f"{MODEL_NAME}_last.pt"
    start_time = time.time()

    # --- RESUME IF POSSIBLE ---
    print("\n[resume] checking for existing training checkpoint...")
    resume = try_load_last_checkpoint(last_ckpt_name, SAVE_DIR)
    if resume is not None:
        vision_model.load_state_dict(resume["vision_model"])
        classifier.load_state_dict(resume["classifier"])
        optimizer.load_state_dict(resume["optimizer"])
        scheduler.load_state_dict(resume["scheduler"])
        scaler.load_state_dict(resume["scaler"])
        start_epoch     = resume["epoch"] + 1
        best_f1         = resume.get("best_f1", 0.0)
        best_epoch      = resume.get("best_epoch", 0)
        patience        = resume.get("patience", 0)
        epoch_logs      = resume.get("epoch_logs", [])
        best_preds      = resume.get("best_preds", [])
        best_gts        = resume.get("best_gts", [])
        best_confs      = resume.get("best_confs", [])
        # RNG state for reproducibility across resumes
        if "rng_torch" in resume:
            torch.set_rng_state(resume["rng_torch"])
        if "rng_cuda" in resume and torch.cuda.is_available():
            try: torch.cuda.set_rng_state_all(resume["rng_cuda"])
            except Exception: pass
        if "rng_numpy" in resume:
            np.random.set_state(resume["rng_numpy"])
        print(f"[resume] starting at epoch {start_epoch+1}/{EPOCHS} (best F1 so far: {best_f1:.4f})")
        del resume
        gc.collect(); torch.cuda.empty_cache()

    for epoch in range(start_epoch, EPOCHS):
        vision_model.train()
        classifier.train()
        total_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"  E{epoch+1}/{EPOCHS} [T]", leave=False)
        for px, y in train_bar:
            px = px.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(backbone_params + head_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)

        vision_model.eval()
        classifier.eval()
        preds, gts, confs = [], [], []
        val_loss = 0.0

        with torch.no_grad(), torch.amp.autocast("cuda"):
            for px, y in val_loader:
                px = px.to(DEVICE, non_blocking=True)
                y_dev = y.to(DEVICE, non_blocking=True)
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                val_loss += criterion(logits, y_dev).item()
                probs = torch.softmax(logits, dim=1)
                max_confs, max_preds = probs.max(dim=1)
                preds.extend(max_preds.cpu().numpy())
                confs_batch = max_confs.cpu().numpy()
                confs.extend(confs_batch)
                gts.extend(y.numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(gts, preds, average="macro")

        epoch_logs.append({
            "epoch": epoch + 1,
            "train_loss": round(avg_loss, 5),
            "val_loss": round(avg_val_loss, 5),
            "val_f1": round(val_f1, 4),
        })

        print(f"  E{epoch+1:03d} | Loss: {avg_loss:.4f}/{avg_val_loss:.4f} | F1: {val_f1:.4f}", end="")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            patience = 0
            best_preds = preds.copy()
            best_gts = gts.copy()
            best_confs = confs.copy()
            ok = save_checkpoint(vision_model, classifier, hidden_size,
                                 best_f1, best_epoch,
                                 f"{MODEL_NAME}_best.pt", model_path)
            print(f" >>> BEST {'✅' if ok else '❌'}")
        else:
            patience += 1
            print(f" ({patience}/{EARLY_STOP_PATIENCE})")
            if patience >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        # --- Save resumable "last" checkpoint every epoch ---
        last_state = {
            "epoch": epoch,
            "vision_model": vision_model.state_dict(),
            "classifier": classifier.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "patience": patience,
            "epoch_logs": epoch_logs,
            "best_preds": best_preds,
            "best_gts": best_gts,
            "best_confs": best_confs,
            "hidden_size": hidden_size,
            "class_to_idx": CLASS_TO_IDX,
            "model_name": MODEL_NAME,
            "label_smoothing": LABEL_SMOOTHING,
            "rng_torch": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "rng_numpy": np.random.get_state(),
        }
        save_last_checkpoint(last_state, last_ckpt_name, SAVE_DIR)
        del last_state

    elapsed = time.time() - start_time
    class_thresholds = {}
    if best_preds and best_gts:
        report = classification_report(best_gts, best_preds,
                                       target_names=list(CLASS_TO_IDX.keys()),
                                       output_dict=True)
        cm = confusion_matrix(best_gts, best_preds)
    else:
        report = {cls: {"f1-score": 0} for cls in CLASS_TO_IDX}
        cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

    print(f"\n{'='*60}")
    print(f"  {MODEL_NAME} DONE")
    print(f"  Best F1: {best_f1:.4f} @ epoch {best_epoch}")
    print(f"  Time: {elapsed/60:.1f} min")
    print(f"{'='*60}")
    print("\nPer-class F1:")
    for cls in CLASS_TO_IDX:
        print(f"  {cls:12s}: {report[cls]['f1-score']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  {'':>12}", end="")
    for name in CLASS_TO_IDX: print(f"{name:>10}", end="")
    print()
    for i, name in enumerate(CLASS_TO_IDX):
        print(f"  {name:>12}", end="")
        for j in range(NUM_CLASSES): print(f"{cm[i][j]:>10}", end="")
        print()

    print(f"\n🔍 Final checkpoint verification:")
    final_ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    print(f"  Checkpoint F1: {final_ckpt['best_f1']:.4f}, Epoch: {final_ckpt['epoch']}")
    assert abs(final_ckpt["best_f1"] - best_f1) < 1e-6, "CHECKPOINT MISMATCH!"
    print(f"  ✅ Verified — checkpoint is correct")

    log_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_log.json")
    with open(log_path, "w") as f:
        json.dump({
            "model": MODEL_NAME,
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "total_epochs": epoch + 1,
            "train_time_min": round(elapsed / 60, 1),
            "label_smoothing": LABEL_SMOOTHING,
            "per_class_f1": {cls: round(report[cls]["f1-score"], 4) for cls in CLASS_TO_IDX},
            "confusion_matrix": cm.tolist(),
            "class_thresholds": class_thresholds,
            "epoch_logs": epoch_logs,
        }, f, indent=2)
    print(f"  Training log saved: {log_path}")

if __name__ == "__main__":
    main()

In [ ]:
"""
============================================================
   STANDALONE TRAINING — ConvNeXt_DINOv3
   Garment Macro Router (Upper/Lower/Underwear/Other)
============================================================
Fixes applied vs previous runs:
  1. LABEL_SMOOTHING = 0.0 (was 0.1 — caused 85%+ predictions below 0.60)
  2. Checkpoint verification after save (prevents silent corruption)
  3. Local-first save then copy to Drive (prevents Drive write crash)
  4. ConvNeXt stage-based unfreezing (was bugged in original script)
"""

import os
import cv2
import numpy as np
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import math
import json
import time
import gc
import shutil
import timm

# ==============================================================
# CONFIG
# ==============================================================

MODEL_NAME = "ConvNeXt_DINOv3"
MODEL_ID = "convnext_base.dinov3_lvd1689m"
IMG_SIZE = 384
BATCH_SIZE = 32
LR_HEAD = 3e-4
LR_BACKBONE = 1e-5
UNFREEZE_LAYERS = 2         # Last 2 stages (stage 2 + stage 3)

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data"
]

SAVE_DIR = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models"
LOCAL_SAVE = "/content"  # Fast local SSD for checkpoint writes


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 60
EARLY_STOP_PATIENCE = 12
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.3
LABEL_SMOOTHING = 0.0      # FIX: was 0.1
NUM_WORKERS = 4

APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING
# ==============================================================
UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# ==============================================================
# PREPROCESSING
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

# ==============================================================
# AUGMENTATIONS + NORMALIZATION
# ==============================================================

_tmp_model = timm.create_model(MODEL_ID, pretrained=False, num_classes=0)
_data_cfg = timm.data.resolve_data_config(model=_tmp_model)
normalize_fn = transforms.Normalize(mean=_data_cfg["mean"], std=_data_cfg["std"])
del _tmp_model

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.3),
    transforms.RandomRotation(degrees=30, fill=128),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10, fill=128),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3, fill=128),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), value=0.5),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ==============================================================
# DATASET
# ==============================================================

class RouterDataset(Dataset):
    def __init__(self, samples, transform, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")
        pixel_values = self.transform(img)
        pixel_values = normalize_fn(pixel_values)
        return pixel_values, label

# ==============================================================
# DATA COLLECTION
# ==============================================================

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

def collect_samples(roots):
    samples = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cls_name in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            label = map_class(cls_name)
            if label is None:
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    samples.append((os.path.join(cls_path, f), label))
    return samples

# ==============================================================
# MODEL
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))

class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        super().__init__(optimizer, last_epoch=-1)
    def get_lr(self):
        step = self.last_epoch
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        return [base_lr * scale for base_lr in self.base_lrs]

def load_model():
    """Load ConvNeXt-Base DINOv3 backbone with stage-based unfreezing."""
    vision_model = timm.create_model(MODEL_ID, pretrained=True, num_classes=0)
    hidden_size = vision_model.num_features

    # Freeze all
    for p in vision_model.parameters():
        p.requires_grad = False

    # ConvNeXt uses .stages (not .blocks like ViT)
    unfrozen = 0
    assert hasattr(vision_model, "stages"), "Expected ConvNeXt to have .stages attribute!"
    n_stages = len(vision_model.stages)
    print(f"  [unfreeze] ConvNeXt: {n_stages} stages, unfreezing last {UNFREEZE_LAYERS}")

    for stage in vision_model.stages[-UNFREEZE_LAYERS:]:
        for p in stage.parameters():
            p.requires_grad = True
            unfrozen += p.numel()

    # Unfreeze head norm, head fc, norm_pre
    if hasattr(vision_model, "head"):
        if hasattr(vision_model.head, "norm"):
            for p in vision_model.head.norm.parameters():
                p.requires_grad = True
                unfrozen += p.numel()
        if hasattr(vision_model.head, "fc"):
            for p in vision_model.head.fc.parameters():
                p.requires_grad = True
                unfrozen += p.numel()
    if hasattr(vision_model, "norm_pre"):
        for p in vision_model.norm_pre.parameters():
            p.requires_grad = True
            unfrozen += p.numel()

    print(f"  [unfreeze] Total unfrozen: {unfrozen:,}")

    # Sanity check
    trainable = sum(p.numel() for p in vision_model.parameters() if p.requires_grad)
    assert trainable > 0, f"FATAL: 0 trainable params in ConvNeXt! unfrozen={unfrozen}"

    return vision_model, hidden_size

def extract_features(model, pixel_values):
    return model(pixel_values)


# ==============================================================
# PER-CLASS THRESHOLD COMPUTATION
# ==============================================================

def compute_class_thresholds(preds, gts, confs, target_precision=0.95):
    """
    For each class, find the minimum confidence threshold where
    precision >= target_precision. This gives per-class rejection
    thresholds for production use.

    Returns dict: {class_name: threshold}
    """
    thresholds = {}
    preds = np.array(preds)
    gts = np.array(gts)
    confs = np.array(confs)

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        # All samples where model predicted this class
        pred_mask = preds == cls_idx
        if pred_mask.sum() == 0:
            thresholds[cls_name] = 0.99  # No predictions → very high threshold
            continue

        cls_confs = confs[pred_mask]
        cls_correct = (gts[pred_mask] == cls_idx)

        # Sort by confidence descending
        sort_idx = np.argsort(-cls_confs)
        cls_confs_sorted = cls_confs[sort_idx]
        cls_correct_sorted = cls_correct[sort_idx]

        # Walk from highest confidence down, find where precision drops below target
        cumulative_correct = np.cumsum(cls_correct_sorted)
        cumulative_total = np.arange(1, len(cls_correct_sorted) + 1)
        precisions = cumulative_correct / cumulative_total

        # Find the lowest confidence where precision >= target
        valid = np.where(precisions >= target_precision)[0]
        if len(valid) > 0:
            last_valid = valid[-1]
            thresholds[cls_name] = float(cls_confs_sorted[last_valid])
        else:
            # Can't reach target precision — use threshold where precision is maximized
            best_idx = np.argmax(precisions)
            thresholds[cls_name] = float(cls_confs_sorted[best_idx])

        # Also compute stats for logging
        correct_confs = cls_confs[cls_correct]
        wrong_confs = cls_confs[~cls_correct]

        n_total = int(pred_mask.sum())
        n_correct = int(cls_correct.sum())
        prec = n_correct / n_total if n_total > 0 else 0

        print(f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} conf_wrong={wrong_confs.mean():.3f}" if len(wrong_confs) > 0
              else f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} (no wrong predictions)")

    return thresholds

# ==============================================================
# CHECKPOINT SAVE + VERIFY
# ==============================================================

def save_checkpoint(vision_model, classifier, hidden_size, best_f1, best_epoch,
                    filename, drive_path):
    local_path = os.path.join(LOCAL_SAVE, filename)
    ckpt = {
        "vision_model": vision_model.state_dict(),
        "classifier": classifier.state_dict(),
        "class_to_idx": CLASS_TO_IDX,
        "hidden_size": hidden_size,
        "best_f1": best_f1,
        "epoch": best_epoch,
        "model_name": MODEL_NAME,
        "label_smoothing": LABEL_SMOOTHING,
    }
    torch.save(ckpt, local_path)

    verify = torch.load(local_path, map_location="cpu", weights_only=False)
    if abs(verify["best_f1"] - best_f1) > 1e-6:
        print(f"  ❌ CHECKPOINT VERIFICATION FAILED!")
        return False
    print(f"  ✅ Checkpoint verified: F1={verify['best_f1']:.4f}, epoch={verify['epoch']}")
    del verify

    try:
        shutil.copy2(local_path, drive_path)
        print(f"  ✅ Copied to Drive: {drive_path}")
    except Exception as e:
        print(f"  ⚠️ Drive copy failed (local copy safe): {e}")
    return True

# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print(f"   TRAINING: {MODEL_NAME}")
    print(f"   Model: {MODEL_ID}")
    print(f"   Label smoothing: {LABEL_SMOOTHING} (fixed — was 0.1)")
    print("=" * 60)
    print(f"Device: {DEVICE}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    print("\nCollecting samples...")
    all_samples = collect_samples(TRAIN_ROOTS)
    print(f"Total: {len(all_samples)}")

    labels = [l for _, l in all_samples]
    train_samples, val_samples = train_test_split(
        all_samples, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

    train_labels = [l for _, l in train_samples]
    train_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    print("\nClass distribution:")
    for i in range(NUM_CLASSES):
        print(f"  {IDX_TO_CLASS[i]}: {train_counts[i]}")

    class_weights = torch.tensor(
        1.0 / (train_counts + 1e-6), dtype=torch.float32, device=DEVICE
    )
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES

    vision_model, hidden_size = load_model()
    vision_model = vision_model.to(DEVICE)

    trainable = sum(p.numel() for p in vision_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in vision_model.parameters())
    print(f"\nParams: {total:,}, Trainable: {trainable:,} ({100*trainable/total:.1f}%)")

    classifier = ClassifierHead(hidden_size, NUM_CLASSES).to(DEVICE)

    train_loader = DataLoader(
        RouterDataset(train_samples, train_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, drop_last=True,
    )
    val_loader = DataLoader(
        RouterDataset(val_samples, val_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True,
    )

    backbone_params = [p for p in vision_model.parameters() if p.requires_grad]
    head_params = list(classifier.parameters())
    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": LR_HEAD},
        {"params": backbone_params, "lr": LR_BACKBONE},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = len(train_loader) * 3
    scheduler = CosineWarmupScheduler(optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda")

    best_f1 = 0.0
    best_epoch = 0
    patience = 0
    epoch_logs = []
    best_preds, best_gts, best_confs = [], [], []

    model_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_best.pt")
    start_time = time.time()

    for epoch in range(EPOCHS):
        vision_model.train()
        classifier.train()
        total_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"  E{epoch+1}/{EPOCHS} [T]", leave=False)
        for px, y in train_bar:
            px = px.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(backbone_params + head_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)

        vision_model.eval()
        classifier.eval()
        preds, gts, confs = [], [], []
        val_loss = 0.0

        with torch.no_grad(), torch.amp.autocast("cuda"):
            for px, y in val_loader:
                px = px.to(DEVICE, non_blocking=True)
                y_dev = y.to(DEVICE, non_blocking=True)
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                val_loss += criterion(logits, y_dev).item()
                probs = torch.softmax(logits, dim=1)
                max_confs, max_preds = probs.max(dim=1)
                preds.extend(max_preds.cpu().numpy())
                confs_batch = max_confs.cpu().numpy()
                confs.extend(confs_batch)
                gts.extend(y.numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(gts, preds, average="macro")

        epoch_logs.append({
            "epoch": epoch + 1,
            "train_loss": round(avg_loss, 5),
            "val_loss": round(avg_val_loss, 5),
            "val_f1": round(val_f1, 4),
        })

        print(f"  E{epoch+1:03d} | Loss: {avg_loss:.4f}/{avg_val_loss:.4f} | F1: {val_f1:.4f}", end="")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            patience = 0
            best_preds = preds.copy()
            best_gts = gts.copy()
            best_confs = confs.copy()
            ok = save_checkpoint(vision_model, classifier, hidden_size,
                                 best_f1, best_epoch,
                                 f"{MODEL_NAME}_best.pt", model_path)
            print(f" >>> BEST {'✅' if ok else '❌'}")
        else:
            patience += 1
            print(f" ({patience}/{EARLY_STOP_PATIENCE})")
            if patience >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    elapsed = time.time() - start_time
    class_thresholds = {}
    if best_preds and best_gts:
        report = classification_report(best_gts, best_preds,
                                       target_names=list(CLASS_TO_IDX.keys()),
                                       output_dict=True)
        cm = confusion_matrix(best_gts, best_preds)
    else:
        report = {cls: {"f1-score": 0} for cls in CLASS_TO_IDX}
        cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

    print(f"\n{'='*60}")
    print(f"  {MODEL_NAME} DONE")
    print(f"  Best F1: {best_f1:.4f} @ epoch {best_epoch}")
    print(f"  Time: {elapsed/60:.1f} min")
    print(f"{'='*60}")
    print("\nPer-class F1:")
    for cls in CLASS_TO_IDX:
        print(f"  {cls:12s}: {report[cls]['f1-score']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  {'':>12}", end="")
    for name in CLASS_TO_IDX: print(f"{name:>10}", end="")
    print()
    for i, name in enumerate(CLASS_TO_IDX):
        print(f"  {name:>12}", end="")
        for j in range(NUM_CLASSES): print(f"{cm[i][j]:>10}", end="")
        print()

    print(f"\n🔍 Final checkpoint verification:")
    final_ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    print(f"  Checkpoint F1: {final_ckpt['best_f1']:.4f}, Epoch: {final_ckpt['epoch']}")
    assert abs(final_ckpt["best_f1"] - best_f1) < 1e-6, "CHECKPOINT MISMATCH!"
    print(f"  ✅ Verified — checkpoint is correct")

    log_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_log.json")
    with open(log_path, "w") as f:
        json.dump({
            "model": MODEL_NAME,
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "total_epochs": epoch + 1,
            "train_time_min": round(elapsed / 60, 1),
            "label_smoothing": LABEL_SMOOTHING,
            "per_class_f1": {cls: round(report[cls]["f1-score"], 4) for cls in CLASS_TO_IDX},
            "confusion_matrix": cm.tolist(),
            "class_thresholds": class_thresholds,
            "epoch_logs": epoch_logs,
        }, f, indent=2)
    print(f"  Training log saved: {log_path}")

if __name__ == "__main__":
    main()

   TRAINING: ConvNeXt_DINOv3
   Model: convnext_base.dinov3_lvd1689m
   Label smoothing: 0.0 (fixed — was 0.1)
Device: cuda

Total: 9666
Train: 7732, Val: 1934

Class distribution:
  Upper: 4131
  Lower: 3069
  Underwear: 335
  Other: 197


model.safetensors:   0%|          | 0.00/350M [00:00<?, ?B/s]

  [unfreeze] ConvNeXt: 4 stages, unfreezing last 2
  [unfreeze] Total unfrozen: 85,395,456

Params: 87,566,464, Trainable: 85,395,456 (97.5%)


  E001 | Loss: 0.9824/0.5568 | F1: 0.7608  ✅ Checkpoint verified: F1=0.7608, epoch=1
  ✅ Copied to Drive: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/ConvNeXt_DINOv3_best.pt
 >>> BEST ✅


  E002 | Loss: 0.5097/0.4896 | F1: 0.8072  ✅ Checkpoint verified: F1=0.8072, epoch=2
  ✅ Copied to Drive: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/ConvNeXt_DINOv3_best.pt
 >>> BEST ✅


...

  E019 | Loss: 0.0710/0.4023 | F1: 0.9455 (8/12)


  E020 | Loss: 0.0745/0.4145 | F1: 0.9321 (9/12)


  E021 | Loss: 0.0632/0.7189 | F1: 0.9152 (10/12)


  E022 | Loss: 0.0851/0.5189 | F1: 0.9276 (11/12)


  E023 | Loss: 0.0572/0.5404 | F1: 0.9361 (12/12)
  Early stopping at epoch 23

  ConvNeXt_DINOv3 DONE
  Best F1: 0.9477 @ epoch 11
  Time: 86.3 min

Per-class F1:
  Upper       : 0.9486
  Lower       : 0.9329
  Underwear   : 0.9704
  Other       : 0.9388

Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       987        42         2         2
         Lower        59       709         0         0
     Underwear         1         0        82         1
         Other         1         1         1        46

🔍 Final checkpoint verification:
  Checkpoint F1: 0.9477, Epoch: 11
  ✅ Verified — checkpoint is correct
  Training log saved: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/ConvNeXt_DINOv3_training_log.json


## Inferencing

In [ ]:
"""
============================================================
   HELD-OUT EVALUATION — 3 Macro Router Models
   Tests on BOTH held-out datasets (never seen during training)
============================================================
Usage (Colab):
    !pip install --upgrade transformers>=4.56.0 timm>=1.0.20 --quiet
    Then run this script.
"""

import os
import cv2
import json
import time
import numpy as np
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score
)
from tqdm import tqdm
import gc

# ==============================================================
# CONFIG
# ==============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# TWO held-out test datasets (NEVER used in training)
EVAL_DATASETS = {
    "classified_images_13_14_test": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Testing_Dataset/classified_images_13_14_test",
    "Careys_labelled_data": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Testing_Dataset/Careys_labelled_data",
}

# Trained model checkpoints
MODEL_DIRS = [
    #"/content/drive/Shareddrives/Garment Type/model_comparison",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models",
]

# Where to save evaluation results
OUTPUT_DIR = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/eval_results"

BATCH_SIZE = 32
NUM_WORKERS = 4

# --- Preprocessing (must match training exactly) ---
APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING — handles ALL folder name variants
# ==============================================================

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# Exhaustive folder-name → macro-class mapping (covers both datasets)
FOLDER_TO_MACRO = {
    # Upper — Title case (Careys_labelled_data)
    "Blazer": "Upper",
    "Jumpers": "Upper",
    "Shirt": "Upper",
    "T-shirt": "Upper",
    "Dresses": "Upper",
    "Fleece": "Upper",
    # Upper — lowercase (classified_images_13_14_test)
    "blazer": "Upper",
    "jumpers": "Upper",
    "shirt": "Upper",
    "t-shirt": "Upper",
    "dresses": "Upper",
    "fleece": "Upper",

    # Lower — Title case
    "Jeans": "Lower",
    "Trousers": "Lower",
    "Jogging_Bottoms": "Lower",
    "Skirts": "Lower",
    # Lower — lowercase
    "jeans": "Lower",
    "trousers": "Lower",
    "jogging_bottoms": "Lower",
    "skirts": "Lower",

    # Underwear — Title case
    "Bra": "Underwear",
    "Knicker": "Underwear",
    # Underwear — lowercase + typo
    "bra": "Underwear",
    "knicker": "Underwear",
    "kincker": "Underwear",   # typo in dataset 1

    # Other (Towel)
    "Towel": "Other",
    "towel": "Other",
}

# Folders to SKIP (not garment classes)
SKIP_FOLDERS = {"13_14th_test", "unknown", "to_test", "SKIP"}

# ==============================================================
# MODEL CONFIGS (only 3 models — no EfficientNetV2)
# ==============================================================

MODEL_CONFIGS = {
    "SigLIP2_SO400M": {
        "model_id": "google/siglip2-so400m-patch14-384",
        "type": "siglip2",
        "img_size": 384,
    },
    "DINOv3_ViT_Base": {
        "model_id": "vit_base_patch16_dinov3.lvd1689m",
        "type": "timm",
        "img_size": 384,
    },
}

# ==============================================================
# PREPROCESSING (exact copy from training)
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped

    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))


# ==============================================================
# DATASET
# ==============================================================

def collect_samples(root):
    """Collect (path, label, original_class) tuples from folder structure."""
    samples = []
    skipped_folders = []
    unmapped_folders = []

    if not os.path.isdir(root):
        print(f"  ❌ Directory not found: {root}")
        return samples

    for folder_name in sorted(os.listdir(root)):
        folder_path = os.path.join(root, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # Skip known non-class folders
        if folder_name in SKIP_FOLDERS:
            skipped_folders.append(folder_name)
            continue

        # Map folder to macro class
        macro_class = FOLDER_TO_MACRO.get(folder_name)
        if macro_class is None:
            unmapped_folders.append(folder_name)
            continue

        label = CLASS_TO_IDX[macro_class]
        count = 0
        for f in os.listdir(folder_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(folder_path, f), label, folder_name))
                count += 1
        print(f"    {folder_name:20s} → {macro_class:10s} ({count} images)")

    if skipped_folders:
        print(f"    ⏭️  Skipped folders: {', '.join(skipped_folders)}")
    if unmapped_folders:
        print(f"    ⚠️  Unmapped folders: {', '.join(unmapped_folders)}")

    return samples


class EvalDataset(Dataset):
    def __init__(self, samples, transform, normalize_fn=None, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.normalize_fn = normalize_fn
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, orig_cls = self.samples[idx]

        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (384, 384), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")

        pixel_values = self.transform(img)
        if self.normalize_fn:
            pixel_values = self.normalize_fn(pixel_values)

        return pixel_values, label, idx


# ==============================================================
# CLASSIFIER HEAD (must match training)
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.fc(self.dropout(x))


# ==============================================================
# MODEL LOADING
# ==============================================================

def load_backbone(config):
    """Load vision backbone architecture (no weights yet)."""
    model_type = config["type"]
    model_id = config["model_id"]

    if model_type == "siglip2":
        try:
            from transformers import Siglip2VisionModel
            vision_model = Siglip2VisionModel.from_pretrained(model_id)
        except Exception:
            try:
                from transformers import SiglipVisionModel
                vision_model = SiglipVisionModel.from_pretrained(model_id)
            except Exception:
                from transformers import AutoModel
                full = AutoModel.from_pretrained(model_id)
                vision_model = full.vision_model
                del full

        hidden_size = vision_model.config.hidden_size
        normalize_fn = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

        def extract_fn(model, pixel_values):
            return model(pixel_values=pixel_values).pooler_output

        return vision_model, hidden_size, normalize_fn, extract_fn

    elif model_type == "timm":
        import timm
        vision_model = timm.create_model(model_id, pretrained=False, num_classes=0)
        hidden_size = vision_model.num_features
        data_cfg = timm.data.resolve_data_config(model=vision_model)
        normalize_fn = transforms.Normalize(mean=data_cfg["mean"], std=data_cfg["std"])

        def extract_fn(model, pixel_values):
            return model(pixel_values)

        return vision_model, hidden_size, normalize_fn, extract_fn


def find_checkpoint(model_name):
    """Search for checkpoint across all model directories."""
    for d in MODEL_DIRS:
        path = os.path.join(d, f"{model_name}_best.pt")
        if os.path.exists(path):
            return path
    return None


def load_checkpoint(model_name, config):
    """Load trained checkpoint. Returns model, classifier, normalize, extract, meta."""
    ckpt_path = find_checkpoint(model_name)
    if ckpt_path is None:
        raise FileNotFoundError(f"Checkpoint not found for {model_name} in: {MODEL_DIRS}")

    vision_model, hidden_size, normalize_fn, extract_fn = load_backbone(config)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if "hidden_size" in ckpt:
        hidden_size = ckpt["hidden_size"]

    vision_model.load_state_dict(ckpt["vision_model"])
    classifier = ClassifierHead(hidden_size, NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])

    meta = {
        "best_f1_train": ckpt.get("best_f1", None),
        "best_epoch": ckpt.get("epoch", None),
        "label_smoothing": ckpt.get("label_smoothing", "unknown"),
        "class_thresholds": ckpt.get("class_thresholds", None),
    }

    return vision_model, classifier, normalize_fn, extract_fn, meta


# ==============================================================
# EVALUATION
# ==============================================================

@torch.no_grad()
def evaluate_model(vision_model, classifier, extract_fn, dataloader):
    """Run inference and return predictions, ground truths, confidences, indices."""
    vision_model.eval()
    classifier.eval()

    all_preds, all_gts, all_confs, all_indices = [], [], [], []

    for batch_pixels, batch_labels, batch_idx in tqdm(dataloader, desc="  Evaluating"):
        batch_pixels = batch_pixels.to(DEVICE)
        features = extract_fn(vision_model, batch_pixels)
        logits = classifier(features)
        probs = torch.softmax(logits, dim=1)
        confs, preds = probs.max(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_gts.extend(batch_labels.numpy())
        all_confs.extend(confs.cpu().numpy())
        all_indices.extend(batch_idx.numpy())

    return (
        np.array(all_preds),
        np.array(all_gts),
        np.array(all_confs),
        np.array(all_indices),
    )


# ==============================================================
# ANALYSIS HELPERS
# ==============================================================

def compute_metrics(preds, gts, class_names):
    f1_macro = f1_score(gts, preds, average="macro")
    f1_weighted = f1_score(gts, preds, average="weighted")
    accuracy = accuracy_score(gts, preds)
    precision = precision_score(gts, preds, average="macro", zero_division=0)
    recall = recall_score(gts, preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(gts, preds, average=None, zero_division=0)
    cm = confusion_matrix(gts, preds, labels=list(range(len(class_names))))

    return {
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "per_class_f1": {class_names[i]: float(per_class_f1[i])
                         for i in range(len(per_class_f1))},
        "confusion_matrix": cm.tolist(),
    }


def per_class_rejection_analysis(preds, gts, confs, class_thresholds):
    """
    Apply per-class thresholds and show what gets rejected vs accepted.
    Returns stats dict.
    """
    if class_thresholds is None:
        return None

    results = {}
    total_accepted = 0
    total_rejected = 0
    total_correct_accepted = 0
    total_correct_rejected = 0

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        threshold = class_thresholds.get(cls_name, 0.60)
        pred_mask = preds == cls_idx
        n_pred = int(pred_mask.sum())

        if n_pred == 0:
            results[cls_name] = {"threshold": threshold, "predictions": 0}
            continue

        cls_confs = confs[pred_mask]
        cls_correct = (gts[pred_mask] == cls_idx)

        accepted = cls_confs >= threshold
        rejected = ~accepted

        n_accepted = int(accepted.sum())
        n_rejected = int(rejected.sum())

        acc_correct = int(cls_correct[accepted].sum()) if n_accepted > 0 else 0
        acc_precision = acc_correct / n_accepted if n_accepted > 0 else 0
        rej_correct = int(cls_correct[rejected].sum()) if n_rejected > 0 else 0

        total_accepted += n_accepted
        total_rejected += n_rejected
        total_correct_accepted += acc_correct
        total_correct_rejected += rej_correct

        results[cls_name] = {
            "threshold": float(threshold),
            "predictions": n_pred,
            "accepted": n_accepted,
            "rejected": n_rejected,
            "accepted_precision": round(acc_precision, 4),
            "rejected_would_be_correct": rej_correct,
        }

    overall_precision = total_correct_accepted / total_accepted if total_accepted > 0 else 0
    overall_rejection_rate = total_rejected / (total_accepted + total_rejected) if (total_accepted + total_rejected) > 0 else 0

    results["_overall"] = {
        "total_accepted": total_accepted,
        "total_rejected": total_rejected,
        "rejection_rate": round(overall_rejection_rate, 4),
        "accepted_precision": round(overall_precision, 4),
        "false_rejections": total_correct_rejected,
    }

    return results


def find_misclassified(preds, gts, confs, indices, samples):
    misclassified = []
    for i in range(len(preds)):
        if preds[i] != gts[i]:
            sample_idx = indices[i]
            path, label, orig_cls = samples[sample_idx]
            misclassified.append({
                "file": os.path.basename(path),
                "folder": orig_cls,
                "true_label": IDX_TO_CLASS[gts[i]],
                "pred_label": IDX_TO_CLASS[preds[i]],
                "confidence": float(confs[i]),
            })
    misclassified.sort(key=lambda x: x["confidence"], reverse=True)
    return misclassified


def print_confusion_matrix(cm, class_names, model_name):
    header = f"{'':>12s}" + "".join(f"{c:>10s}" for c in class_names)
    print(f"\n  {model_name} Confusion Matrix:")
    print(f"  {header}")
    for i, row in enumerate(cm):
        row_str = f"  {class_names[i]:>12s}" + "".join(f"{v:>10d}" for v in row)
        print(row_str)


# ==============================================================
# EVALUATE ONE DATASET
# ==============================================================

def evaluate_dataset(dataset_name, dataset_root, models_loaded):
    """Evaluate all models on one dataset. Returns results dict."""

    print(f"\n{'='*80}")
    print(f"   DATASET: {dataset_name}")
    print(f"   Path: {dataset_root}")
    print(f"{'='*80}")

    # Collect samples
    print(f"\n  Collecting samples...")
    samples = collect_samples(dataset_root)
    print(f"\n  Total samples: {len(samples)}")

    if len(samples) == 0:
        print(f"  ❌ No samples found!")
        return None

    # Distribution
    dist = defaultdict(int)
    sub_dist = defaultdict(int)
    for _, label, orig_cls in samples:
        dist[IDX_TO_CLASS[label]] += 1
        sub_dist[orig_cls] += 1

    print(f"\n  Macro class distribution:")
    for cls_name in CLASS_TO_IDX:
        print(f"    {cls_name:12s}: {dist[cls_name]:5d}")

    class_names = list(CLASS_TO_IDX.keys())
    dataset_results = {}

    for model_name, model_bundle in models_loaded.items():
        vision_model, classifier, normalize_fn, extract_fn, meta, config = model_bundle

        print(f"\n  {'─'*60}")
        print(f"  📊 {model_name} on {dataset_name}")
        print(f"  {'─'*60}")

        # Build dataloader
        val_transform = transforms.Compose([
            transforms.Resize((config["img_size"], config["img_size"])),
            transforms.ToTensor(),
        ])
        dataset = EvalDataset(samples, val_transform, normalize_fn, APPLY_PREPROCESSING)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS,
                                pin_memory=True)

        # Run inference
        t_start = time.perf_counter()
        preds, gts, confs, indices = evaluate_model(
            vision_model, classifier, extract_fn, dataloader
        )
        t_infer = time.perf_counter() - t_start
        fps = len(samples) / t_infer

        # Metrics
        metrics = compute_metrics(preds, gts, class_names)
        misclassified = find_misclassified(preds, gts, confs, indices, samples)

        # Confidence stats
        correct_mask = preds == gts
        mean_conf_correct = float(confs[correct_mask].mean()) if correct_mask.sum() > 0 else 0
        mean_conf_wrong = float(confs[~correct_mask].mean()) if (~correct_mask).sum() > 0 else 0

        # Per-class confidence
        per_class_conf = {}
        for cls_idx, cls_name in IDX_TO_CLASS.items():
            cls_mask = gts == cls_idx
            if cls_mask.sum() > 0:
                per_class_conf[cls_name] = {
                    "mean_conf": float(confs[cls_mask].mean()),
                    "min_conf": float(confs[cls_mask].min()),
                    "accuracy": float(correct_mask[cls_mask].mean() * 100),
                }

        # Per-class threshold rejection analysis
        class_thresholds = meta.get("class_thresholds")
        rejection = per_class_rejection_analysis(preds, gts, confs, class_thresholds)

        # Print results
        print(f"\n  F1 (macro):     {metrics['f1_macro']:.4f}")
        print(f"  F1 (weighted):  {metrics['f1_weighted']:.4f}")
        print(f"  Accuracy:       {metrics['accuracy']:.4f}")
        print(f"  Precision:      {metrics['precision_macro']:.4f}")
        print(f"  Recall:         {metrics['recall_macro']:.4f}")
        print(f"  Inference:      {t_infer:.1f}s ({fps:.1f} img/s)")

        print(f"\n  Per-class F1:")
        for cls_name in class_names:
            f1_val = metrics["per_class_f1"].get(cls_name, 0)
            ci = per_class_conf.get(cls_name, {})
            print(f"    {cls_name:12s}: F1={f1_val:.4f}  "
                  f"conf={ci.get('mean_conf',0):.3f} (min={ci.get('min_conf',0):.3f})  "
                  f"acc={ci.get('accuracy',0):.1f}%")

        print_confusion_matrix(metrics["confusion_matrix"], class_names, model_name)

        print(f"\n  Confidence:")
        print(f"    Correct: {mean_conf_correct:.4f}  |  Wrong: {mean_conf_wrong:.4f}")
        print(f"    Misclassified: {len(misclassified)} ({len(misclassified)/len(samples)*100:.1f}%)")

        # Per-class threshold rejection
        if rejection:
            print(f"\n  📏 Per-class threshold rejection (from training):")
            for cls_name in class_names:
                r = rejection.get(cls_name, {})
                if r.get("predictions", 0) > 0:
                    print(f"    {cls_name:12s}: thresh={r['threshold']:.3f}  "
                          f"accepted={r['accepted']}/{r['predictions']}  "
                          f"precision={r['accepted_precision']:.3f}  "
                          f"false_rej={r['rejected_would_be_correct']}")

            ov = rejection["_overall"]
            print(f"    {'OVERALL':12s}: accepted={ov['total_accepted']}  "
                  f"rejected={ov['total_rejected']} ({ov['rejection_rate']*100:.1f}%)  "
                  f"precision={ov['accepted_precision']:.3f}  "
                  f"false_rej={ov['false_rejections']}")
        else:
            print(f"\n  ⚠️  No per-class thresholds in checkpoint (old model?)")

        # Worst mistakes
        if misclassified:
            print(f"\n  🔴 Top 15 worst mistakes:")
            for m in misclassified[:15]:
                print(f"    {m['file']:40s} | {m['folder']:15s} → "
                      f"true={m['true_label']:10s} pred={m['pred_label']:10s} "
                      f"conf={m['confidence']:.3f}")

        dataset_results[model_name] = {
            "metrics": metrics,
            "inference_time_s": float(t_infer),
            "fps": float(fps),
            "confidence": {
                "mean_correct": mean_conf_correct,
                "mean_wrong": mean_conf_wrong,
                "per_class": per_class_conf,
            },
            "class_thresholds_used": class_thresholds,
            "rejection_analysis": rejection,
            "misclassified_count": len(misclassified),
            "misclassified_top20": misclassified[:20],
        }

    return {
        "dataset_name": dataset_name,
        "dataset_root": dataset_root,
        "total_samples": len(samples),
        "class_distribution": dict(dist),
        "subclass_distribution": dict(sub_dist),
        "models": dataset_results,
    }


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 80)
    print("   HELD-OUT EVALUATION — 3 Models × 2 Datasets")
    print("=" * 80)
    print(f"Device: {DEVICE}")
    print(f"Datasets: {list(EVAL_DATASETS.keys())}")
    print(f"Models: {list(MODEL_CONFIGS.keys())}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ----------------------------------------------------------
    # 1. Load all models ONCE (reuse across datasets)
    # ----------------------------------------------------------
    print(f"\n{'='*80}")
    print("   LOADING MODELS")
    print(f"{'='*80}")

    models_loaded = {}
    for model_name, config in MODEL_CONFIGS.items():
        ckpt_path = find_checkpoint(model_name)
        if ckpt_path is None:
            print(f"\n  ⚠️  SKIPPING {model_name} — checkpoint not found")
            continue

        print(f"\n  Loading {model_name} from {ckpt_path}...")
        try:
            vision_model, classifier, normalize_fn, extract_fn, meta = \
                load_checkpoint(model_name, config)
            vision_model = vision_model.to(DEVICE)
            classifier = classifier.to(DEVICE)

            print(f"    Train F1: {meta['best_f1_train']}, Epoch: {meta['best_epoch']}, "
                  f"Label smoothing: {meta['label_smoothing']}")
            if meta['class_thresholds']:
                print(f"    Per-class thresholds: {meta['class_thresholds']}")
            else:
                print(f"    ⚠️  No per-class thresholds in checkpoint")

            models_loaded[model_name] = (
                vision_model, classifier, normalize_fn, extract_fn, meta, config
            )
        except Exception as e:
            print(f"    ❌ Failed: {e}")
            import traceback
            traceback.print_exc()

    if not models_loaded:
        print("\n❌ No models loaded!")
        return

    # ----------------------------------------------------------
    # 2. Evaluate on each dataset
    # ----------------------------------------------------------
    all_dataset_results = {}

    for ds_name, ds_path in EVAL_DATASETS.items():
        result = evaluate_dataset(ds_name, ds_path, models_loaded)
        if result:
            all_dataset_results[ds_name] = result

    # ----------------------------------------------------------
    # 3. Cross-dataset comparison table
    # ----------------------------------------------------------
    print(f"\n\n{'='*100}")
    print(f"   CROSS-DATASET COMPARISON")
    print(f"{'='*100}")

    header = (f"{'Model':24s} │ {'Dataset':36s} │ "
              f"{'F1':>6s} {'Acc':>6s} {'Err':>5s} │ "
              f"{'Upper':>7s} {'Lower':>7s} {'Undwr':>7s} {'Other':>7s} │ "
              f"{'Rej%':>5s} {'APrc':>5s}")
    print(header)
    print("─" * 100)

    for model_name in MODEL_CONFIGS:
        if model_name not in models_loaded:
            continue
        for ds_name, ds_result in all_dataset_results.items():
            if model_name not in ds_result["models"]:
                continue
            mr = ds_result["models"][model_name]
            m = mr["metrics"]
            pcf = m["per_class_f1"]

            rej_rate = ""
            a_prec = ""
            if mr.get("rejection_analysis") and "_overall" in mr["rejection_analysis"]:
                ov = mr["rejection_analysis"]["_overall"]
                rej_rate = f"{ov['rejection_rate']*100:.1f}"
                a_prec = f"{ov['accepted_precision']:.3f}"

            print(f"{model_name:24s} │ {ds_name:36s} │ "
                  f"{m['f1_macro']:>6.4f} {m['accuracy']:>6.4f} {mr['misclassified_count']:>5d} │ "
                  f"{pcf.get('Upper',0):>7.4f} {pcf.get('Lower',0):>7.4f} "
                  f"{pcf.get('Underwear',0):>7.4f} {pcf.get('Other',0):>7.4f} │ "
                  f"{rej_rate:>5s} {a_prec:>5s}")

    print("─" * 100)

    # ----------------------------------------------------------
    # 4. Generalization gap
    # ----------------------------------------------------------
    print(f"\n  Generalization gap (train F1 → held-out F1):")
    for model_name in models_loaded:
        meta = models_loaded[model_name][4]
        train_f1 = meta["best_f1_train"]
        for ds_name, ds_result in all_dataset_results.items():
            if model_name in ds_result["models"]:
                held_f1 = ds_result["models"][model_name]["metrics"]["f1_macro"]
                gap = (train_f1 - held_f1) if train_f1 else None
                if gap is not None:
                    ind = "🟢" if gap < 0.03 else ("🟡" if gap < 0.06 else "🔴")
                    print(f"    {ind} {model_name:24s} on {ds_name:30s}: "
                          f"{train_f1:.4f} → {held_f1:.4f}  (gap={gap:+.4f})")

    # ----------------------------------------------------------
    # 5. Save results
    # ----------------------------------------------------------
    results_path = os.path.join(OUTPUT_DIR, "eval_all_results.json")

    save_data = {}
    for ds_name, ds_result in all_dataset_results.items():
        save_data[ds_name] = ds_result.copy()

    with open(results_path, "w") as f:
        json.dump(save_data, f, indent=2, default=str)
    print(f"\n✅ All results saved to: {results_path}")

    # Save misclassified per model per dataset
    misc_path = os.path.join(OUTPUT_DIR, "misclassified_all.json")
    misc_data = {}
    for ds_name, ds_result in all_dataset_results.items():
        misc_data[ds_name] = {}
        for model_name, mr in ds_result["models"].items():
            misc_data[ds_name][model_name] = mr.get("misclassified_top20", [])
    with open(misc_path, "w") as f:
        json.dump(misc_data, f, indent=2)
    print(f"✅ Misclassified details saved to: {misc_path}")

    # Cleanup
    for model_name, bundle in models_loaded.items():
        del bundle
    torch.cuda.empty_cache()
    gc.collect()


if __name__ == "__main__":
    main()

   HELD-OUT EVALUATION — 3 Models × 2 Datasets
Device: cuda
Datasets: ['classified_images_13_14_test', 'Careys_labelled_data']
Models: ['SigLIP2_SO400M', 'DINOv3_ViT_Base']

   LOADING MODELS

  Loading SigLIP2_SO400M from /content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/SigLIP2_SO400M_best.pt...


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

You are using a model of type siglip_vision_model to instantiate a model of type siglip2_vision_model. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

...


  F1 (macro):     0.9371
  F1 (weighted):  0.9457
  Accuracy:       0.9457
  Precision:      0.9307
  Recall:         0.9445
  Inference:      36.5s (36.3 img/s)

  Per-class F1:
    Upper       : F1=0.9446  conf=0.994 (min=0.518)  acc=96.1%
    Lower       : F1=0.9481  conf=0.987 (min=0.510)  acc=92.6%
    Underwear   : F1=0.9524  conf=0.999 (min=0.968)  acc=98.8%
    Other       : F1=0.9032  conf=0.999 (min=0.988)  acc=90.3%

  DINOv3_ViT_Base Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       597        16         5         3
         Lower        43       548         1         0
     Underwear         1         0        80         0
         Other         2         0         1        28

  Confidence:
    Correct: 0.9952  |  Wrong: 0.9242
    Misclassified: 72 (5.4%)

  ⚠️  No per-class thresholds in checkpoint (old model?)

  🔴 Top 15 worst mistakes:
    20251214_120511_398_garment_masked.jpg   | fleece          → true=Upper      pred=Ot

  Evaluating: 100%|██████████| 15/15 [03:08<00:00, 12.59s/it]



  F1 (macro):     0.8713
  F1 (weighted):  0.8846
  Accuracy:       0.8820
  Precision:      0.8213
  Recall:         0.9491
  Inference:      188.8s (2.5 img/s)

  Per-class F1:
    Upper       : F1=0.8991  conf=0.701 (min=0.333)  acc=82.8%
    Lower       : F1=0.8693  conf=0.827 (min=0.458)  acc=96.9%
    Underwear   : F1=0.8000  conf=0.930 (min=0.556)  acc=100.0%
    Other       : F1=0.9167  conf=0.831 (min=0.671)  acc=100.0%

  SigLIP2_SO400M Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       245        33        16         2
         Lower         4       123         0         0
     Underwear         0         0        32         0
         Other         0         0         0        11

  Confidence:
    Correct: 0.7711  |  Wrong: 0.6269
    Misclassified: 55 (11.8%)

  ⚠️  No per-class thresholds in checkpoint (old model?)

  🔴 Top 15 worst mistakes:
    20251214_175107_637_garment_img.png      | Dresses         → true=Upper      pred=

  Evaluating: 100%|██████████| 15/15 [00:12<00:00,  1.20it/s]



  F1 (macro):     0.8586
  F1 (weighted):  0.8765
  Accuracy:       0.8734
  Precision:      0.8485
  Recall:         0.8997
  Inference:      12.5s (37.2 img/s)

  Per-class F1:
    Upper       : F1=0.8925  conf=0.968 (min=0.543)  acc=82.8%
    Lower       : F1=0.8612  conf=0.990 (min=0.508)  acc=95.3%
    Underwear   : F1=0.7805  conf=1.000 (min=1.000)  acc=100.0%
    Other       : F1=0.9000  conf=0.996 (min=0.975)  acc=81.8%

  DINOv3_ViT_Base Confusion Matrix:
                   Upper     Lower Underwear     Other
         Upper       245        33        18         0
         Lower         6       121         0         0
     Underwear         0         0        32         0
         Other         2         0         0         9

  Confidence:
    Correct: 0.9859  |  Wrong: 0.9155
    Misclassified: 59 (12.7%)

  ⚠️  No per-class thresholds in checkpoint (old model?)

  🔴 Top 15 worst mistakes:
    20251214_165943_806_garment_img.png      | Dresses         → true=Upper      pred=

In [ ]:
import torch

path = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/SigLIP2_SO400M_best.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
print(f"F1: {ckpt['best_f1']:.4f}, Epoch: {ckpt['epoch']}")

F1: 0.8857, Epoch: 1


## ONNX conversion and inferencing

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import timm
import time
!pip install onnx onnxruntime-gpu --quiet
# ==============================================================
# CONFIG
# ==============================================================

CHECKPOINT_PATH = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_best.pt"
ONNX_OUTPUT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx"

MODEL_ID = "vit_base_patch16_dinov3.lvd1689m"
IMG_SIZE = 384
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPSET_VERSION = 17

# ==============================================================
# WRAPPER MODEL (backbone + classifier + softmax in one)
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))


class DINOv3ForExport(nn.Module):
    """Single module: ViT backbone → classifier → softmax probabilities."""
    def __init__(self, vision_model, classifier):
        super().__init__()
        self.vision_model = vision_model
        self.classifier = classifier

    def forward(self, pixel_values):
        features = self.vision_model(pixel_values)
        logits = self.classifier(features)
        probs = torch.softmax(logits, dim=1)
        return probs


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  ONNX EXPORT — DINOv3_ViT_Base")
    print("=" * 60)

    # 1. Load checkpoint
    print(f"\nLoading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    hidden_size = ckpt["hidden_size"]
    print(f"  F1: {ckpt['best_f1']:.4f}, Epoch: {ckpt['epoch']}, Hidden: {hidden_size}")

    # 2. Build model
    print(f"\nBuilding model architecture...")
    vision_model = timm.create_model(MODEL_ID, pretrained=False, num_classes=0)
    vision_model.load_state_dict(ckpt["vision_model"])

    classifier = ClassifierHead(hidden_size, NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])

    export_model = DINOv3ForExport(vision_model, classifier)
    export_model = export_model.to(DEVICE)
    export_model.eval()

    total_params = sum(p.numel() for p in export_model.parameters())
    print(f"  Total params: {total_params:,}")

    # 3. Test PyTorch inference
    print(f"\nTesting PyTorch inference...")
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.no_grad():
        pt_output = export_model(dummy_input)
    print(f"  Output shape: {pt_output.shape}")
    print(f"  Sum of probs: {pt_output.sum().item():.4f} (should be 1.0)")
    print(f"  Probs: {pt_output.cpu().numpy().round(4)}")

    # 4. Export to ONNX
    # 4. Export to ONNX (TensorRT-safe STATIC graph)
    print(f"\nExporting to ONNX (STATIC TensorRT-safe, opset={OPSET_VERSION})...")
    os.makedirs(os.path.dirname(ONNX_OUTPUT), exist_ok=True)

    torch.onnx.export(
        export_model,
        dummy_input,
        ONNX_OUTPUT,
        opset_version=OPSET_VERSION,

        input_names=["pixel_values"],
        output_names=["probabilities"],

        # Force static inference graph
        training=torch.onnx.TrainingMode.EVAL,
        do_constant_folding=True,

        # Use legacy exporter (more stable for TRT)
        dynamo=False,

        # Prevent weird shape tensors
        keep_initializers_as_inputs=False,
    )


    onnx_size_mb = os.path.getsize(ONNX_OUTPUT) / (1024 * 1024)
    print(f"  ✅ Saved: {ONNX_OUTPUT}")
    print(f"  File size: {onnx_size_mb:.1f} MB")

    # 5. Validate ONNX model
    print(f"\nValidating ONNX model...")
    import onnx
    onnx_model = onnx.load(ONNX_OUTPUT)
    onnx.checker.check_model(onnx_model)
    print(f"  ✅ ONNX model is valid")

    # 6. Test ONNX Runtime inference
    print(f"\nTesting ONNX Runtime inference...")
    import onnxruntime as ort

    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    session = ort.InferenceSession(ONNX_OUTPUT, providers=providers)
    actual_provider = session.get_providers()[0]
    print(f"  Provider: {actual_provider}")

    input_np = dummy_input.cpu().numpy()
    ort_output = session.run(None, {"pixel_values": input_np})[0]

    print(f"  ONNX output shape: {ort_output.shape}")
    print(f"  ONNX probs: {ort_output.round(4)}")

    # 7. Compare PyTorch vs ONNX
    pt_np = pt_output.cpu().numpy()
    # Line 145: relax tolerance

    max_diff = np.abs(pt_np - ort_output).max()
    print(f"\n  Max difference (PT vs ONNX): {max_diff:.8f}")
    assert max_diff < 1e-3, f"MISMATCH! max_diff={max_diff}"
    print(f"  ✅ PyTorch and ONNX outputs match")

    # 8. Benchmark ONNX speed
    print(f"\nBenchmarking ONNX speed (100 runs)...")
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        session.run(None, {"pixel_values": input_np})
        times.append(time.perf_counter() - t0)
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms
    print(f"  Avg: {avg_ms:.1f} ms/image ({fps:.1f} FPS)")

    print(f"\n{'='*60}")
    print(f"  DONE — Ready for TensorRT conversion on Orin")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  ONNX EXPORT — DINOv3_ViT_Base

Loading checkpoint: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_best.pt
  F1: 0.9474, Epoch: 19, Hidden: 768

Building model architecture...
  Total params: 85,644,292

Testing PyTorch inference...
  Output shape: torch.Size([1, 4])
  Sum of probs: 1.0000 (should be 1.0)
  Probs: [[0.5568 0.286  0.1208 0.0364]]

Exporting to ONNX (STATIC TensorRT-safe, opset=17)...


/tmp/ipython-input-2846824074.py:103: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:2185: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message
/usr/

  ✅ Saved: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_remove_if.onnx
  File size: 327.0 MB

Validating ONNX model...
  ✅ ONNX model is valid

Testing ONNX Runtime inference...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


  Provider: CPUExecutionProvider
  ONNX output shape: (1, 4)
  ONNX probs: [[0.5568 0.286  0.1208 0.0364]]

  Max difference (PT vs ONNX): 0.00000066
  ✅ PyTorch and ONNX outputs match

Benchmarking ONNX speed (100 runs)...
  Avg: 377.5 ms/image (2.6 FPS)

  DONE — Ready for TensorRT conversion on Orin


In [ ]:
!python -m onnxsim \
"/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_remove_if.onnx" \
"/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_remove_if_sim.onnx"


Your model contains "Tile" ops or/and "ConstantOfShape" ops. Folding these ops 
can make the simplified model much larger. If it is not expected, please specify
"--no-large-tensor" (which will lose some optimization chances)
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                    ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Abs                │ 1              │ 0                │
│ Add                │ 121            │ 96               │
│ Cast               │ 26             │ 0                │
│ Concat             │ 75             │ 49               │
│ Constant           │ 777            │ 183              │
│ ConstantOfShape    │ 2              │ 0                │
│ Conv               │ 1              │ 1                │
│ Cos                │ 1              │ 0                │
│ Div                │ 50             │ 12               │
│ Equal             

In [ ]:
"""
============================================================
   ONNX EXPORT — SigLIP2_SO400M (FIXED)
   Converts .pt checkpoint → .onnx for TensorRT deployment
============================================================
Usage (Colab):
    !pip install --upgrade transformers>=4.56.0 onnx onnxruntime-gpu --quiet
    Then run this cell.

FIXES vs previous version:
  1. Use SAME dummy_input (batch=1) for PT test, ONNX export, and validation
  2. Removed dynamic_axes (TRT uses fixed batch=1, matches DINOv3 export)
  3. Proper PT vs ONNX comparison on identical input
"""

import os
import torch
import torch.nn as nn
import numpy as np
import time

# !pip install onnx onnxruntime

# ==============================================================
# CONFIG
# ==============================================================

CHECKPOINT_PATH = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M_best.pt"
ONNX_OUTPUT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx"

MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPSET_VERSION = 17

# ==============================================================
# WRAPPER MODEL
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))


class SigLIP2ForExport(nn.Module):
    """
    Single module: SigLIP2 vision backbone → pooler → classifier → softmax.
    Wraps the HuggingFace model so ONNX export sees a clean forward(pixel_values) → probs.
    """
    def __init__(self, vision_model, classifier):
        super().__init__()
        self.vision_model = vision_model
        self.classifier = classifier

    def forward(self, pixel_values):
        outputs = self.vision_model(pixel_values=pixel_values)
        features = outputs.pooler_output
        logits = self.classifier(features)
        probs = torch.softmax(logits, dim=1)
        return probs


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  ONNX EXPORT — SigLIP2_SO400M (FIXED)")
    print("=" * 60)

    # 1. Load checkpoint
    print(f"\nLoading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    hidden_size = ckpt["hidden_size"]
    print(f"  F1: {ckpt['best_f1']:.4f}, Epoch: {ckpt['epoch']}, Hidden: {hidden_size}")

    # 2. Build model architecture
    print(f"\nLoading SigLIP2 vision backbone...")
    try:
        from transformers import Siglip2VisionModel
        vision_model = Siglip2VisionModel.from_pretrained(MODEL_ID)
        print(f"  Loaded via Siglip2VisionModel")
    except Exception:
        try:
            from transformers import SiglipVisionModel
            vision_model = SiglipVisionModel.from_pretrained(MODEL_ID)
            print(f"  Loaded via SiglipVisionModel")
        except Exception:
            from transformers import AutoModel
            full = AutoModel.from_pretrained(MODEL_ID)
            vision_model = full.vision_model
            del full
            print(f"  Loaded via AutoModel fallback")

    # Load trained weights
    vision_model.load_state_dict(ckpt["vision_model"])

    classifier = ClassifierHead(hidden_size, NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])

    export_model = SigLIP2ForExport(vision_model, classifier)
    export_model = export_model.to(DEVICE)
    export_model.eval()

    total_params = sum(p.numel() for p in export_model.parameters())
    print(f"  Total params: {total_params:,}")

    # 3. Create ONE dummy input — reuse for PT test, ONNX export, and validation
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    # 4. Test PyTorch inference
    print(f"\nTesting PyTorch inference...")
    with torch.no_grad():
        pt_output = export_model(dummy_input)
    print(f"  Output shape: {pt_output.shape}")
    print(f"  Sum of probs: {pt_output.sum().item():.4f} (should be 1.0)")
    print(f"  Probs: {pt_output.cpu().numpy().round(4)}")

    # 5. Export to ONNX (STATIC batch=1, same as DINOv3 export)
    print(f"\nExporting to ONNX (STATIC batch=1, opset={OPSET_VERSION})...")
    print(f"  (SigLIP2 is 428M params — this may take 1-2 minutes)")
    os.makedirs(os.path.dirname(ONNX_OUTPUT), exist_ok=True)

    torch.onnx.export(
        export_model,
        dummy_input,                    # batch=1 — SAME input used for PT test
        ONNX_OUTPUT,
        opset_version=OPSET_VERSION,
        input_names=["pixel_values"],
        output_names=["probabilities"],
        # No dynamic_axes — TRT uses fixed batch=1 (matches DINOv3 export)
        training=torch.onnx.TrainingMode.EVAL,
        do_constant_folding=True,
        dynamo=False,
        keep_initializers_as_inputs=False,
    )

    onnx_size_mb = os.path.getsize(ONNX_OUTPUT) / (1024 * 1024)
    print(f"  ✅ Saved: {ONNX_OUTPUT}")
    print(f"  File size: {onnx_size_mb:.1f} MB")

    # 6. Validate ONNX model
    print(f"\nValidating ONNX model...")
    import onnx
    onnx_model = onnx.load(ONNX_OUTPUT)
    onnx.checker.check_model(onnx_model)
    print(f"  ✅ ONNX model is valid")

    # 7. Test ONNX Runtime inference — SAME input as PyTorch
    print(f"\nTesting ONNX Runtime inference...")
    import onnxruntime as ort

    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    session = ort.InferenceSession(ONNX_OUTPUT, providers=providers)
    actual_provider = session.get_providers()[0]
    print(f"  Provider: {actual_provider}")

    input_np = dummy_input.cpu().numpy()                          # SAME input
    ort_output = session.run(None, {"pixel_values": input_np})[0]

    print(f"  ONNX output shape: {ort_output.shape}")
    print(f"  ONNX probs: {ort_output.round(4)}")

    # 8. Compare PyTorch vs ONNX — now comparing SAME input → SAME expected output
    pt_np = pt_output.cpu().numpy()
    max_diff = np.abs(pt_np - ort_output).max()
    print(f"\n  Max difference (PT vs ONNX): {max_diff:.8f}")
    assert max_diff < 1e-3, f"MISMATCH! max_diff={max_diff}"
    print(f"  ✅ PyTorch and ONNX outputs match (diff < 1e-3)")

    # 9. Benchmark ONNX speed
    print(f"\nBenchmarking ONNX speed (50 runs)...")
    for _ in range(5):
        session.run(None, {"pixel_values": input_np})
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        session.run(None, {"pixel_values": input_np})
        times.append(time.perf_counter() - t0)
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms
    print(f"  Avg: {avg_ms:.1f} ms/image ({fps:.1f} FPS)")

    print(f"\n{'='*60}")
    print(f"  DONE — Ready for TensorRT conversion on Orin")
    print(f"  trtexec --onnx=SigLIP2_SO400M.onnx --saveEngine=SigLIP2_SO400M_fp16.engine --fp16 --memPoolSize=workspace:8192M")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  ONNX EXPORT — SigLIP2_SO400M (FIXED)

Loading checkpoint: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M_best.pt
  F1: 0.9538, Epoch: 37, Hidden: 1152

Loading SigLIP2 vision backbone...


You are using a model of type siglip_vision_model to instantiate a model of type siglip2_vision_model. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Siglip2VisionModel LOAD REPORT from: google/siglip2-so400m-patch14-384
Key                                                          | Status     |                                                                                                 
-------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------
text_model.encoder.layers.{0...26}.self_attn.k_proj.bias     | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.self_attn.out_proj.bias   | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.layer_norm1.weight        | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.self_attn.k_proj.weight   | U

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip2-so400m-patch14-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...26}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.q_pro

  Loaded via SiglipVisionModel
  Total params: 428,230,212

Testing PyTorch inference...
  Output shape: torch.Size([1, 4])
  Sum of probs: 1.0000 (should be 1.0)
  Probs: [[0.4057 0.494  0.0373 0.063 ]]

Exporting to ONNX (STATIC batch=1, opset=17)...
  (SigLIP2 is 428M params — this may take 1-2 minutes)


/tmp/ipykernel_1048/23533982.py:130: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


  ✅ Saved: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx
  File size: 1634.0 MB

Validating ONNX model...
  ✅ ONNX model is valid

Testing ONNX Runtime inference...
  Provider: CPUExecutionProvider
  ONNX output shape: (1, 4)
  ONNX probs: [[0.4056 0.4942 0.0373 0.063 ]]

  Max difference (PT vs ONNX): 0.00014922
  ✅ PyTorch and ONNX outputs match (diff < 1e-3)

Benchmarking ONNX speed (50 runs)...
  Avg: 2169.0 ms/image (0.5 FPS)

  DONE — Ready for TensorRT conversion on Orin
  trtexec --onnx=SigLIP2_SO400M.onnx --saveEngine=SigLIP2_SO400M_fp16.engine --fp16 --memPoolSize=workspace:8192M


In [ ]:
"""
============================================================
   ONNX INFERENCE VERIFICATION — Both Models
   Tests DINOv3 and SigLIP2 .onnx on real garment images
============================================================
Usage (Colab):
    !pip install onnxruntime-gpu --quiet
    Then run this cell.
"""

import os
import cv2
import json
import numpy as np
from PIL import Image
import time
import onnxruntime as ort

# ==============================================================
# CONFIG
# ==============================================================

DINOV3_ONNX = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx"
SIGLIP2_ONNX = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx"

# Production configs (for thresholds)
DINOV3_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/dinov3_production_config.json"
SIGLIP2_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/siglip2_production_config.json"

# Test images — pick a folder with known labels
TEST_DIR = "/content/drive/Shareddrives/Garment Type/classified_images_13_14_test"
MAX_TEST_IMAGES = 1300  # Test on first N images for quick verification

IMG_SIZE = 384

CLASS_NAMES = ["Upper", "Lower", "Underwear", "Other"]

# Normalisation per model
DINOV3_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
DINOV3_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

SIGLIP2_MEAN = np.array([0.5, 0.5, 0.5], dtype=np.float32).reshape(3, 1, 1)
SIGLIP2_STD = np.array([0.5, 0.5, 0.5], dtype=np.float32).reshape(3, 1, 1)

# ==============================================================
# PREPROCESSING (exact match to training)
# ==============================================================

BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

FOLDER_TO_MACRO = {
    "blazer": "Upper", "jumpers": "Upper", "shirt": "Upper",
    "t-shirt": "Upper", "dresses": "Upper", "fleece": "Upper",
    "jeans": "Lower", "trousers": "Lower", "jogging_bottoms": "Lower",
    "skirts": "Lower", "bra": "Underwear", "kincker": "Underwear",
    "knicker": "Underwear", "towel": "Other",
    "Blazer": "Upper", "Jumpers": "Upper", "Shirt": "Upper",
    "T-shirt": "Upper", "Dresses": "Upper", "Fleece": "Upper",
    "Jeans": "Lower", "Trousers": "Lower", "Jogging_Bottoms": "Lower",
    "Skirts": "Lower", "Bra": "Underwear", "Knicker": "Underwear",
    "Towel": "Other",
}

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
SKIP_FOLDERS = {"13_14th_test", "unknown", "to_test", "SKIP"}


def preprocess_crop(img_np):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > BLACK_THRESH, 255, 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)
    gray_bg = np.full_like(img_np, BACKGROUND_COLOR, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)
    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
        return square
    mx, my = int(bw * MARGIN_RATIO), int(bh * MARGIN_RATIO)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]
    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    return square


def prepare_input(img_path, mean, std):
    """Full pipeline: read → preprocess → resize → normalize → NCHW float32."""
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None

    # Preprocess (gray bg, pad-to-square)
    square_bgr = preprocess_crop(img_bgr)

    # Resize
    resized = cv2.resize(square_bgr, (IMG_SIZE, IMG_SIZE))

    # BGR → RGB → float [0,1] → CHW
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    chw = np.transpose(rgb, (2, 0, 1))  # HWC → CHW

    # Normalize
    chw = (chw - mean) / std

    # Add batch dim
    return chw[np.newaxis, ...]  # (1, 3, 384, 384)


# ==============================================================
# COLLECT TEST SAMPLES
# ==============================================================

def collect_test_samples(root, max_samples=200):
    samples = []
    for folder_name in sorted(os.listdir(root)):
        folder_path = os.path.join(root, folder_name)
        if not os.path.isdir(folder_path):
            continue
        if folder_name in SKIP_FOLDERS:
            continue
        macro = FOLDER_TO_MACRO.get(folder_name)
        if macro is None:
            continue
        label = CLASS_TO_IDX[macro]
        for f in sorted(os.listdir(folder_path)):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(folder_path, f), label, folder_name))
                if len(samples) >= max_samples:
                    return samples
    return samples


# ==============================================================
# RUN INFERENCE
# ==============================================================

def run_onnx_eval(session, samples, mean, std, model_name, thresholds=None):
    """Run ONNX model on samples and print results."""
    correct = 0
    total = 0
    errors = []
    accepted = 0
    accepted_correct = 0
    rejected = 0
    times = []

    for path, label, folder in samples:
        input_np = prepare_input(path, mean, std)
        if input_np is None:
            continue

        t0 = time.perf_counter()
        probs = session.run(None, {"pixel_values": input_np})[0][0]
        times.append(time.perf_counter() - t0)

        pred = int(np.argmax(probs))
        conf = float(probs[pred])
        total += 1

        if pred == label:
            correct += 1

        # Threshold check
        if thresholds:
            thresh = thresholds.get(CLASS_NAMES[pred], 0.5)
            if conf >= thresh:
                accepted += 1
                if pred == label:
                    accepted_correct += 1
            else:
                rejected += 1

        if pred != label:
            errors.append({
                "file": os.path.basename(path),
                "folder": folder,
                "true": CLASS_NAMES[label],
                "pred": CLASS_NAMES[pred],
                "conf": conf,
            })

    accuracy = correct / total if total > 0 else 0
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms

    print(f"\n  📊 {model_name} — ONNX Inference Results")
    print(f"  {'─'*50}")
    print(f"  Accuracy:    {accuracy:.4f} ({correct}/{total})")
    print(f"  Errors:      {len(errors)}")
    print(f"  Speed:       {avg_ms:.1f} ms/img ({fps:.1f} FPS)")

    if thresholds:
        acc_prec = accepted_correct / accepted if accepted > 0 else 0
        print(f"\n  Threshold rejection:")
        print(f"    Accepted:  {accepted} ({accepted/total*100:.1f}%)")
        print(f"    Rejected:  {rejected} ({rejected/total*100:.1f}%) → 'Unknown'")
        print(f"    Accepted precision: {acc_prec:.4f}")

    if errors:
        print(f"\n  Top 10 mistakes (sorted by confidence):")
        errors.sort(key=lambda x: x["conf"], reverse=True)
        for e in errors[:10]:
            print(f"    {e['file']:40s} | {e['folder']:15s} "
                  f"true={e['true']:10s} pred={e['pred']:10s} conf={e['conf']:.3f}")

    return accuracy, fps


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  ONNX INFERENCE VERIFICATION — Both Models")
    print("=" * 60)

    # Collect test samples
    print(f"\nCollecting test samples from: {TEST_DIR}")
    samples = collect_test_samples(TEST_DIR, MAX_TEST_IMAGES)
    print(f"  Loaded {len(samples)} samples")

    if len(samples) == 0:
        print("❌ No samples found!")
        return

    results = {}

    # ── DINOv3 ──
    if os.path.exists(DINOV3_ONNX):
        print(f"\n{'='*60}")
        print(f"  Loading DINOv3 ONNX: {DINOV3_ONNX}")
        size_mb = os.path.getsize(DINOV3_ONNX) / (1024 * 1024)
        print(f"  Size: {size_mb:.1f} MB")

        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        session = ort.InferenceSession(DINOV3_ONNX, providers=providers)
        print(f"  Provider: {session.get_providers()[0]}")

        # Load thresholds
        dinov3_thresh = None
        if os.path.exists(DINOV3_CONFIG):
            with open(DINOV3_CONFIG) as f:
                dinov3_thresh = json.load(f).get("class_thresholds")
            print(f"  Thresholds: {dinov3_thresh}")

        acc, fps = run_onnx_eval(session, samples, DINOV3_MEAN, DINOV3_STD,
                                  "DINOv3_ViT_Base", dinov3_thresh)
        results["DINOv3"] = {"accuracy": acc, "fps": fps}
        del session
    else:
        print(f"\n⚠️  DINOv3 ONNX not found: {DINOV3_ONNX}")

    # ── SigLIP2 ──
    if os.path.exists(SIGLIP2_ONNX):
        print(f"\n{'='*60}")
        print(f"  Loading SigLIP2 ONNX: {SIGLIP2_ONNX}")
        size_mb = os.path.getsize(SIGLIP2_ONNX) / (1024 * 1024)
        print(f"  Size: {size_mb:.1f} MB")

        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        session = ort.InferenceSession(SIGLIP2_ONNX, providers=providers)
        print(f"  Provider: {session.get_providers()[0]}")

        # Load thresholds
        siglip2_thresh = None
        if os.path.exists(SIGLIP2_CONFIG):
            with open(SIGLIP2_CONFIG) as f:
                siglip2_thresh = json.load(f).get("class_thresholds")
            print(f"  Thresholds: {siglip2_thresh}")

        acc, fps = run_onnx_eval(session, samples, SIGLIP2_MEAN, SIGLIP2_STD,
                                  "SigLIP2_SO400M", siglip2_thresh)
        results["SigLIP2"] = {"accuracy": acc, "fps": fps}
        del session
    else:
        print(f"\n⚠️  SigLIP2 ONNX not found: {SIGLIP2_ONNX}")

    # ── Comparison ──
    if len(results) == 2:
        print(f"\n\n{'='*60}")
        print(f"  SIDE-BY-SIDE COMPARISON (ONNX)")
        print(f"{'='*60}")
        print(f"  {'Model':20s} {'Accuracy':>10s} {'FPS':>10s} {'Size':>10s}")
        print(f"  {'─'*50}")
        for name, r in results.items():
            onnx_path = DINOV3_ONNX if name == "DINOv3" else SIGLIP2_ONNX
            size = os.path.getsize(onnx_path) / (1024 * 1024)
            print(f"  {name:20s} {r['accuracy']:>10.4f} {r['fps']:>9.1f}x {size:>9.1f}MB")


if __name__ == "__main__":
    main()

  ONNX INFERENCE VERIFICATION — Both Models

  Loaded 1300 samples

  Loading DINOv3 ONNX: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx
  Size: 327.0 MB
  Provider: CUDAExecutionProvider
  Thresholds: {'Upper': 0.8047, 'Lower': 0.7486, 'Underwear': 0.4247, 'Other': 0.615}

  📊 DINOv3_ViT_Base — ONNX Inference Results
  ──────────────────────────────────────────────────
  Accuracy:    0.9146 (1189/1300)
  Errors:      111
  Speed:       9.3 ms/img (107.4 FPS)

  Threshold rejection:
    Accepted:  1242 (95.5%)
    Rejected:  58 (4.5%) → 'Unknown'
    Accepted precision: 0.9324

  Top 10 mistakes (sorted by confidence):
    20251210_153020_976_garment_masked.jpg   | jogging_bottoms true=Lower      pred=Upper      conf=1.000
    20251212_204440_772_garment_masked.jpg   | jogging_bottoms true=Lower      pred=Upper      conf=1.000
    20251210_153456_317_garment_masked.jpg   | jogging_bottoms true=Lower      pred=Upper      conf=1.000


In [ ]:
DINOV3_ONNX = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx"
SIGLIP2_ONNX = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx"


import onnx
# DINOv3 — needs .data file in same dir
m = onnx.load("/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx", load_external_data=True)
print(f"DINOv3 inputs: {[i.name for i in m.graph.input]}")
print(f"DINOv3 outputs: {[o.name for o in m.graph.output]}")

# SigLIP2 — self-contained
m2 = onnx.load("/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx")
print(f"SigLIP2 inputs: {[i.name for i in m2.graph.input]}")
print(f"SigLIP2 outputs: {[o.name for o in m2.graph.output]}")

DINOv3 inputs: ['pixel_values']
DINOv3 outputs: ['probabilities']
SigLIP2 inputs: ['pixel_values']
SigLIP2 outputs: ['probabilities']


In [ ]:
"""
Fix DINOv3 ONNX for TensorRT — fold If nodes safely
"""

!pip install -U onnx==1.16.1 onnxsim==0.4.36 onnxruntime

import onnx
from onnx import shape_inference
from onnxsim import simplify
import numpy as np
import os

ONNX_PATH = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx"
ONNX_OUT  = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_sim.onnx"

print("Loading ONNX model...")
model = onnx.load(ONNX_PATH, load_external_data=True)

# -------------------------------------------------
# 1. Run shape inference FIRST (critical for ViT/DINO)
# -------------------------------------------------
print("Running ONNX shape inference...")
model = shape_inference.infer_shapes(model)

# -------------------------------------------------
# 2. Count If nodes before
# -------------------------------------------------
if_count = sum(1 for n in model.graph.node if n.op_type == "If")
print(f"If nodes before simplification: {if_count}")

# -------------------------------------------------
# 3. Simplify safely (NO deprecated args)
# -------------------------------------------------
print("Running onnx-simplifier (fixed shape 1x3x384x384)...")

model_sim, check = simplify(
    model,
    overwrite_input_shapes={"pixel_values": [1, 3, 384, 384]},
    dynamic_input_shape=False,
    skip_constant_folding=True,      # <-- avoids undefined tensor bug
)

assert check, "Simplified model validation failed!"

# -------------------------------------------------
# 4. Count If nodes after
# -------------------------------------------------
if_count_after = sum(1 for n in model_sim.graph.node if n.op_type == "If")
print(f"If nodes after simplification: {if_count_after}")

# -------------------------------------------------
# 5. Save
# -------------------------------------------------
onnx.save(model_sim, ONNX_OUT)
size_mb = os.path.getsize(ONNX_OUT) / (1024 * 1024)
print(f"Saved: {ONNX_OUT} ({size_mb:.1f} MB)")

# -------------------------------------------------
# 6. Verify with ONNX Runtime
# -------------------------------------------------
print("Testing inference with ONNX Runtime...")

import onnxruntime as ort

session = ort.InferenceSession(
    ONNX_OUT,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)

dummy = np.random.randn(1, 3, 384, 384).astype(np.float32)
out = session.run(None, {"pixel_values": dummy})[0]

print(f"ORT inference OK — output shape: {out.shape}, sum: {out.sum():.4f}")


Loading ONNX model...
Running ONNX shape inference...
If nodes before simplification: 2
Running onnx-simplifier (fixed shape 1x3x384x384)...
If nodes after simplification: 2
Saved: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_sim.onnx (327.0 MB)
Testing inference with ONNX Runtime...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


ORT inference OK — output shape: (1, 4), sum: 1.0000


In [ ]:
import onnx
import numpy as np
from onnx import helper, numpy_helper

IN  = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_sim.onnx"
OUT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_no_if.onnx"

!pip install onnx onnxruntime onnxsim polygraphy


print("Loading model...")
model = onnx.load(IN)
graph = model.graph

new_nodes = []
removed = 0

for node in graph.node:
    if node.op_type == "If":
        removed += 1
        print("Removing If:", node.name)

        # Try safe replacement:
        # If output is int64 → constant [1,2]
        # else → identity from first available input tensor
        if len(node.output) == 1:
            out = node.output[0]

            # Default constant for shape branches
            const = np.array([1, 2], dtype=np.int64)
            tensor = numpy_helper.from_array(const, name=f"const_fix_{removed}")

            const_node = helper.make_node(
                "Constant",
                inputs=[],
                outputs=[out],
                value=tensor,
                name=f"TRT_REMOVE_IF_{removed}"
            )
            new_nodes.append(const_node)
        else:
            # fallback: skip (should not happen)
            continue
    else:
        new_nodes.append(node)

graph.ClearField("node")
graph.node.extend(new_nodes)

onnx.save(model, OUT)
print(f"Saved model without If nodes → {OUT}")
print(f"Total If removed: {removed}")


Loading model...
Removing If: /vision_model/If
Removing If: /vision_model/If_1
Saved model without If nodes → /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_no_if.onnx
Total If removed: 2


In [ ]:
!python -m polygraphy.tools.surgeon.sanitize \
  "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx" \
  --fold-constants \
  --external-data-dir "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th" \
  --output "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_clean.onnx"


/usr/bin/python3: No module named polygraphy.tools.surgeon.sanitize


In [ ]:
!pip show polygraphy
!pip install onnx-graphsurgeon


Name: polygraphy
Version: 0.49.26
Summary: Polygraphy: A Deep Learning Inference Prototyping and Debugging Toolkit
Home-page: https://github.com/NVIDIA/TensorRT/tree/main/tools/Polygraphy
Author: NVIDIA
Author-email: svc_tensorrt@nvidia.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.2 MB/s eta 0:00:00


In [ ]:
!polygraphy surgeon sanitize \
  "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx" \
  --fold-constants \
  --external-data-dir "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th" \
  --output "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_clean.onnx"


[W] 'colored' module is not installed, will not use colors when logging. To enable colors, please install the 'colored' module: python3 -m pip install colored
[I] RUNNING | Command: /usr/local/bin/polygraphy surgeon sanitize /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx --fold-constants --external-data-dir /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th --output /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_clean.onnx
[I] Loading model: /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx
[W] Falling back to `onnx.shape_inference` because `onnxruntime.tools.symbolic_shape_infer` either could not be loaded or did not run successfully.
    Note that using ONNX-Runtime for shape inference may be faster and require less memory.
    Consider installing ONNX-Runtime or setting POLYGRAPHY_AUTOINSTALL_DEPS=1 in 

In [ ]:
"""
Run in Colab — check if onnx-simplifier broke DINOv3
"""
import numpy as np
import onnxruntime as ort

ORIGINAL = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx"
SIMPLIFIED = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_remove_if_sim.onnx"

CLASS_NAMES = ["Upper", "Lower", "Underwear", "Other"]

# 10 random inputs
np.random.seed(42)
dummy_inputs = [np.random.randn(1, 3, 384, 384).astype(np.float32) for _ in range(10)]

sess_orig = ort.InferenceSession(ORIGINAL, providers=['CUDAExecutionProvider'])
sess_sim  = ort.InferenceSession(SIMPLIFIED, providers=['CUDAExecutionProvider'])

print("Comparing Original vs Simplified ONNX (10 random inputs):\n")
max_diff_all = 0
for i, inp in enumerate(dummy_inputs):
    out_orig = sess_orig.run(None, {"pixel_values": inp})[0][0]
    out_sim  = sess_sim.run(None, {"pixel_values": inp})[0][0]
    diff = np.abs(out_orig - out_sim).max()
    max_diff_all = max(max_diff_all, diff)
    pred_orig = CLASS_NAMES[np.argmax(out_orig)]
    pred_sim  = CLASS_NAMES[np.argmax(out_sim)]
    match = "✅" if pred_orig == pred_sim else "❌"
    print(f"  Input {i}: orig={pred_orig}({out_orig.max():.3f})  "
          f"sim={pred_sim}({out_sim.max():.3f})  diff={diff:.6f} {match}")

print(f"\nMax diff across all: {max_diff_all:.6f}")
if max_diff_all > 0.01:
    print("❌ SIMPLIFIED ONNX IS CORRUPTED — weights didn't transfer properly")
    print("   Re-run onnx-simplifier with load_external_data=True")
else:
    print("✅ Simplified ONNX matches original")

Comparing Original vs Simplified ONNX (10 random inputs):

  Input 0: orig=Upper(0.571)  sim=Upper(0.571)  diff=0.000000 ✅
  Input 1: orig=Upper(0.559)  sim=Upper(0.559)  diff=0.000000 ✅
  Input 2: orig=Upper(0.554)  sim=Upper(0.554)  diff=0.000000 ✅
  Input 3: orig=Upper(0.543)  sim=Upper(0.543)  diff=0.000000 ✅
  Input 4: orig=Upper(0.560)  sim=Upper(0.560)  diff=0.000000 ✅
  Input 5: orig=Upper(0.552)  sim=Upper(0.552)  diff=0.000000 ✅
  Input 6: orig=Upper(0.545)  sim=Upper(0.545)  diff=0.000000 ✅
  Input 7: orig=Upper(0.567)  sim=Upper(0.567)  diff=0.000000 ✅
  Input 8: orig=Upper(0.560)  sim=Upper(0.560)  diff=0.000000 ✅
  Input 9: orig=Upper(0.569)  sim=Upper(0.569)  diff=0.000000 ✅

Max diff across all: 0.000000
✅ Simplified ONNX matches original


In [ ]:
"""
============================================================
   TEMPERATURE SCALING — DINOv3 + SigLIP2
   Finds optimal T per model on validation set.
   No retraining — just calibration post-hoc.
============================================================

Run in Colab:
    !pip install timm transformers --quiet
    Then run this cell.

After running, update production config JSONs with the
temperature values and use: probs = softmax(logits / T)
"""

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize_scalar
from tqdm import tqdm
import timm
import json

# ==============================================================
# CONFIG
# ==============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DINOV3_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_best.pt"
SIGLIP2_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M_best.pt"

DINOV3_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/dinov3_production_config.json"
SIGLIP2_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/siglip2_production_config.json"

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
]

IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_CLASSES = 4

BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING
# ==============================================================

UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

# ==============================================================
# PREPROCESSING (exact training match)
# ==============================================================

def preprocess_crop(img_np):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > BLACK_THRESH, 255, 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)
    gray_bg = np.full_like(img_np, BACKGROUND_COLOR, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)
    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
        from PIL import Image
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))
    mx, my = int(bw * MARGIN_RATIO), int(bh * MARGIN_RATIO)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]
    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    from PIL import Image
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))


class ValDataset(Dataset):
    def __init__(self, samples, transform, normalize):
        self.samples = samples
        self.transform = transform
        self.normalize = normalize

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            from PIL import Image
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        else:
            img = preprocess_crop(img_bgr)
        px = self.transform(img)
        px = self.normalize(px)
        return px, label


# ==============================================================
# CLASSIFIER HEAD
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))


# ==============================================================
# COLLECT LOGITS FROM MODEL
# ==============================================================

def collect_logits(vision_model, classifier, loader):
    """Run inference, collect raw LOGITS (before softmax) and true labels."""
    all_logits = []
    all_labels = []

    vision_model.eval()
    classifier.eval()

    with torch.no_grad(), torch.amp.autocast("cuda"):
        for px, labels in tqdm(loader, desc="  Collecting logits"):
            px = px.to(DEVICE)
            feats = vision_model(px)

            # Handle HuggingFace models that return objects
            if hasattr(feats, 'pooler_output'):
                feats = feats.pooler_output
            elif hasattr(feats, 'last_hidden_state'):
                feats = feats.last_hidden_state[:, 0]

            logits = classifier(feats)  # RAW logits, no softmax
            all_logits.append(logits.cpu())
            all_labels.append(labels)

    return torch.cat(all_logits), torch.cat(all_labels)


# ==============================================================
# TEMPERATURE SCALING
# ==============================================================

def find_optimal_temperature(logits, labels, t_range=(0.5, 10.0)):
    """
    Find T that minimizes Negative Log Likelihood on calibration set.
    NLL = -mean(log(softmax(logits/T)[correct_class]))
    """
    def nll_at_t(T):
        scaled = logits / T
        log_probs = F.log_softmax(scaled, dim=1)
        nll = F.nll_loss(log_probs, labels)
        return nll.item()

    result = minimize_scalar(nll_at_t, bounds=t_range, method='bounded')
    return result.x, result.fun


def compute_ece(probs, labels, n_bins=15):
    """Expected Calibration Error."""
    confidences = probs.max(dim=1).values
    predictions = probs.argmax(dim=1)
    accuracies = predictions.eq(labels)

    ece = 0.0
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].float().mean()
        bin_conf = confidences[mask].mean()
        bin_size = mask.sum().float() / len(labels)
        ece += (bin_acc - bin_conf).abs() * bin_size

    return ece.item()


def analyze_calibration(logits, labels, T, model_name):
    """Show detailed before/after calibration stats."""

    print(f"\n  {'─'*55}")
    print(f"  {model_name} — Temperature = {T:.3f}")
    print(f"  {'─'*55}")

    # Before (T=1)
    probs_before = F.softmax(logits, dim=1)
    preds_before = probs_before.argmax(dim=1)
    confs_before = probs_before.max(dim=1).values
    correct_mask = preds_before.eq(labels)

    # After (T=optimal)
    probs_after = F.softmax(logits / T, dim=1)
    preds_after = probs_after.argmax(dim=1)
    confs_after = probs_after.max(dim=1).values

    # Accuracy unchanged (T doesn't change argmax)
    acc = correct_mask.float().mean().item()
    print(f"  Accuracy: {acc:.4f} (unchanged by temperature)")

    # ECE
    ece_before = compute_ece(probs_before, labels)
    ece_after = compute_ece(probs_after, labels)
    print(f"\n  ECE (Expected Calibration Error):")
    print(f"    Before (T=1):   {ece_before:.4f}")
    print(f"    After  (T={T:.2f}): {ece_after:.4f}  ({'✅ improved' if ece_after < ece_before else '⚠️ worse'})")

    # Confidence distribution
    print(f"\n  Confidence Distribution:")
    print(f"    {'':15s} {'Before (T=1)':>15s} {'After (T='+f'{T:.2f}'+')':>15s}")

    c_correct_before = confs_before[correct_mask]
    c_wrong_before = confs_before[~correct_mask]
    c_correct_after = confs_after[correct_mask]
    c_wrong_after = confs_after[~correct_mask]

    print(f"    {'Correct mean':15s} {c_correct_before.mean():.3f}           {c_correct_after.mean():.3f}")
    print(f"    {'Wrong mean':15s} {c_wrong_before.mean():.3f}           {c_wrong_after.mean():.3f}")
    print(f"    {'Gap':15s} {(c_correct_before.mean()-c_wrong_before.mean()):.3f}           {(c_correct_after.mean()-c_wrong_after.mean()):.3f}")

    # Confidence percentiles
    print(f"\n  Confidence Percentiles (ALL predictions):")
    for p in [10, 25, 50, 75, 90]:
        # Line 265-266: cast to float
        b = torch.quantile(confs_before.float(), p/100).item()
        a = torch.quantile(confs_after.float(), p/100).item()
        print(f"    P{p:02d}: {b:.3f} → {a:.3f}")

    # Per-class analysis
    print(f"\n  Per-Class Confidence (After T={T:.2f}):")
    for cls_idx, cls_name in IDX_TO_CLASS.items():
        cls_mask = labels == cls_idx
        if cls_mask.sum() == 0:
            continue
        pred_mask = preds_after == cls_idx
        cls_correct = (preds_after[cls_mask] == cls_idx)
        cls_confs = confs_after[cls_mask]
        correct_confs = cls_confs[cls_correct]
        wrong_confs = cls_confs[~cls_correct]

        print(f"    {cls_name:12s}: n={cls_mask.sum():4d}  "
              f"acc={cls_correct.float().mean():.3f}  "
              f"conf_correct={correct_confs.mean():.3f}  "
              + (f"conf_wrong={wrong_confs.mean():.3f}" if len(wrong_confs) > 0 else "no errors"))

    # Threshold effectiveness after scaling
    print(f"\n  Threshold Analysis (After T={T:.2f}):")
    for thresh in [0.50, 0.60, 0.70, 0.80, 0.90]:
        accepted = confs_after >= thresh
        n_accepted = accepted.sum().item()
        if n_accepted == 0:
            continue
        acc_accepted = correct_mask[accepted].float().mean().item()
        reject_pct = (1 - n_accepted / len(labels)) * 100
        print(f"    thresh={thresh:.2f}: accept={n_accepted:4d} ({100-reject_pct:.1f}%)  "
              f"precision={acc_accepted:.4f}  reject={reject_pct:.1f}%")

    return T


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  TEMPERATURE SCALING — DINOv3 + SigLIP2")
    print("  (No retraining — just calibration)")
    print("=" * 60)

    # ── Collect val samples (same split as training) ──
    print("\nCollecting samples...")
    all_samples = []
    for root in TRAIN_ROOTS:
        if not os.path.isdir(root):
            continue
        for cls_name in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            label = map_class(cls_name)
            if label is None:
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    all_samples.append((os.path.join(cls_path, f), label))

    print(f"  Total samples: {len(all_samples)}")
    labels = [l for _, l in all_samples]
    _, val_samples = train_test_split(all_samples, test_size=0.2, stratify=labels, random_state=42)
    print(f"  Val split: {len(val_samples)} (same random_state=42 as training)")

    results = {}

    # ==============================================================
    # DINOv3
    # ==============================================================
    if os.path.exists(DINOV3_CHECKPOINT):
        print(f"\n{'='*60}")
        print(f"  DINOv3_ViT_Base")
        print(f"{'='*60}")

        ckpt = torch.load(DINOV3_CHECKPOINT, map_location=DEVICE, weights_only=False)
        hidden_size = ckpt["hidden_size"]
        print(f"  Loaded: F1={ckpt['best_f1']:.4f}, epoch={ckpt['epoch']}")

        vision_model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=False, num_classes=0)
        vision_model.load_state_dict(ckpt["vision_model"])
        vision_model = vision_model.to(DEVICE).eval()

        classifier = ClassifierHead(hidden_size, NUM_CLASSES)
        classifier.load_state_dict(ckpt["classifier"])
        classifier = classifier.to(DEVICE).eval()

        # Normalization from timm
        _cfg = timm.data.resolve_data_config(model=vision_model)
        normalize = transforms.Normalize(mean=_cfg["mean"], std=_cfg["std"])
        transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

        loader = DataLoader(ValDataset(val_samples, transform, normalize),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        print(f"  Collecting logits on {len(val_samples)} val samples...")
        logits, labels_t = collect_logits(vision_model, classifier, loader)
        print(f"  Logits shape: {logits.shape}")

        # Find optimal T
        print(f"\n  Optimizing temperature...")
        T_opt, nll_opt = find_optimal_temperature(logits, labels_t)
        print(f"  Optimal T = {T_opt:.4f} (NLL = {nll_opt:.4f})")

        analyze_calibration(logits, labels_t, T_opt, "DINOv3")
        results["DINOv3"] = T_opt

        # Cleanup GPU
        del vision_model, classifier
        torch.cuda.empty_cache()

    # ==============================================================
    # SigLIP2
    # ==============================================================
    if os.path.exists(SIGLIP2_CHECKPOINT):
        print(f"\n{'='*60}")
        print(f"  SigLIP2_SO400M")
        print(f"{'='*60}")

        ckpt = torch.load(SIGLIP2_CHECKPOINT, map_location=DEVICE, weights_only=False)
        hidden_size = ckpt["hidden_size"]
        print(f"  Loaded: F1={ckpt['best_f1']:.4f}, epoch={ckpt['epoch']}")

        # Load SigLIP2 backbone
        try:
            from transformers import Siglip2VisionModel
            vision_model = Siglip2VisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
        except Exception:
            try:
                from transformers import SiglipVisionModel
                vision_model = SiglipVisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
            except Exception:
                from transformers import AutoModel
                full = AutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
                vision_model = full.vision_model
                del full

        vision_model.load_state_dict(ckpt["vision_model"])
        vision_model = vision_model.to(DEVICE).eval()

        classifier = ClassifierHead(hidden_size, NUM_CLASSES)
        classifier.load_state_dict(ckpt["classifier"])
        classifier = classifier.to(DEVICE).eval()

        # SigLIP2 normalization: mean=0.5, std=0.5
        normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

        loader = DataLoader(ValDataset(val_samples, transform, normalize),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        print(f"  Collecting logits on {len(val_samples)} val samples...")
        logits, labels_t = collect_logits(vision_model, classifier, loader)
        print(f"  Logits shape: {logits.shape}")

        print(f"\n  Optimizing temperature...")
        T_opt, nll_opt = find_optimal_temperature(logits, labels_t)
        print(f"  Optimal T = {T_opt:.4f} (NLL = {nll_opt:.4f})")

        analyze_calibration(logits, labels_t, T_opt, "SigLIP2")
        results["SigLIP2"] = T_opt

        del vision_model, classifier
        torch.cuda.empty_cache()

    # ==============================================================
    # SAVE TO CONFIG
    # ==============================================================

    print(f"\n\n{'='*60}")
    print(f"  RESULTS SUMMARY")
    print(f"{'='*60}")

    for model_name, T in results.items():
        print(f"  {model_name}: T = {T:.4f}")

    # Update DINOv3 config
    if "DINOv3" in results and os.path.exists(DINOV3_CONFIG):
        with open(DINOV3_CONFIG, "r") as f:
            cfg = json.load(f)
        cfg["temperature"] = round(results["DINOv3"], 4)
        with open(DINOV3_CONFIG, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"\n  ✅ Updated {DINOV3_CONFIG} with temperature={cfg['temperature']}")

    # Update SigLIP2 config
    if "SigLIP2" in results and os.path.exists(SIGLIP2_CONFIG):
        with open(SIGLIP2_CONFIG, "r") as f:
            cfg = json.load(f)
        cfg["temperature"] = round(results["SigLIP2"], 4)
        with open(SIGLIP2_CONFIG, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"  ✅ Updated {SIGLIP2_CONFIG} with temperature={cfg['temperature']}")

    print(f"\n  Production usage:")
    print(f"  ─────────────────")
    print(f"  config = json.load(open('dinov3_production_config.json'))")
    print(f"  T = config['temperature']")
    print(f"  # After TRT inference gives raw logits:")
    print(f"  scaled = logits / T")
    print(f"  probs = softmax(scaled)")
    print(f"  # Now confidence values are properly calibrated!")

    print(f"\n{'='*60}")
    print(f"  DONE")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  TEMPERATURE SCALING — DINOv3 + SigLIP2
  (No retraining — just calibration)

  Total samples: 9666
  Val split: 1934 (same random_state=42 as training)

  DINOv3_ViT_Base
  Loaded: F1=0.9474, epoch=19


  Logits shape: torch.Size([1934, 4])

  Optimizing temperature...
  Optimal T = 1.8306 (NLL = 0.1855)

  ───────────────────────────────────────────────────────
  DINOv3 — Temperature = 1.831
  ───────────────────────────────────────────────────────
  Accuracy: 0.9338 (unchanged by temperature)

  ECE (Expected Calibration Error):
    Before (T=1):   0.0428
    After  (T=1.83): 0.0164  (✅ improved)

  Confidence Distribution:
                       Before (T=1)  After (T=1.83)
    Correct mean    0.982           0.952
    Wrong mean      0.849           0.758
    Gap             0.133           0.194

  Confidence Percentiles (ALL predictions):
    P10: 0.946 → 0.809
    P25: 0.998 → 0.953
    P50: 1.000 → 0.984
    P75: 1.000 → 0.992
    P90: 1.000 → 0.995

  Per-Class Confidence (After T=1.83):
    Upper       : n=1033  acc=0.949  conf_correct=0.951  conf_wrong=0.728
    Lower       : n= 768  acc=0.909  conf_correct=0.952  conf_wrong=0.769
    Underwear   : n=  84  acc=0.952  conf_c

You are using a model of type siglip_vision_model to instantiate a model of type siglip2_vision_model. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Siglip2VisionModel LOAD REPORT from: google/siglip2-so400m-patch14-384
Key                                                          | Status     |                                                                                                 
-------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------
text_model.encoder.layers.{0...26}.layer_norm1.weight        | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.layer_norm2.bias          | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.mlp.fc2.bias              | U

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip2-so400m-patch14-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...26}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.k_pro

  Logits shape: torch.Size([1934, 4])

  Optimizing temperature...
  Optimal T = 2.6698 (NLL = 0.1851)

  ───────────────────────────────────────────────────────
  SigLIP2 — Temperature = 2.670
  ───────────────────────────────────────────────────────
  Accuracy: 0.9411 (unchanged by temperature)

  ECE (Expected Calibration Error):
    Before (T=1):   0.0508
    After  (T=2.67): 0.0141  (✅ improved)

  Confidence Distribution:
                       Before (T=1)  After (T=2.67)
    Correct mean    0.993           0.958
    Wrong mean      0.939           0.804
    Gap             0.053           0.154

  Confidence Percentiles (ALL predictions):
    P10: 0.997 → 0.864
    P25: 1.000 → 0.960
    P50: 1.000 → 0.982
    P75: 1.000 → 0.990
    P90: 1.000 → 0.994

  Per-Class Confidence (After T=2.67):
    Upper       : n=1033  acc=0.941  conf_correct=0.958  conf_wrong=0.792
    Lower       : n= 768  acc=0.939  conf_correct=0.956  conf_wrong=0.823
    Underwear   : n=  84  acc=0.940  conf_

In [ ]:
"""
Recompute per-class thresholds WITH temperature scaling.
Run AFTER temperature_scaling.py has saved T values to configs.
"""
import os, json, cv2, torch, timm
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DEVICE = "cuda"
IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_CLASSES = 4
TARGET_PRECISION = 0.95  # Same as before

BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

DINOV3_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base_best.pt"
SIGLIP2_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M_best.pt"
DINOV3_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/dinov3_production_config.json"
SIGLIP2_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/siglip2_production_config.json"

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
]

UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]
CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

def preprocess_crop(img_np):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > BLACK_THRESH, 255, 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)
    gray_bg = np.full_like(img_np, BACKGROUND_COLOR, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)
    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        from PIL import Image
        return Image.fromarray(np.full((IMG_SIZE, IMG_SIZE, 3), 128, dtype=np.uint8))
    mx, my = int(bw * MARGIN_RATIO), int(bh * MARGIN_RATIO)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]
    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    from PIL import Image
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

class ValDataset(Dataset):
    def __init__(self, samples, transform, normalize):
        self.samples = samples
        self.transform = transform
        self.normalize = normalize
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            from PIL import Image
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        else:
            img = preprocess_crop(img_bgr)
        return self.normalize(self.transform(img)), label

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x): return self.fc(self.dropout(x))


def collect_logits(vision_model, classifier, loader):
    all_logits, all_labels = [], []
    vision_model.eval(); classifier.eval()
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for px, labels in tqdm(loader, desc="  Collecting logits"):
            feats = vision_model(px.to(DEVICE))
            if hasattr(feats, 'pooler_output'): feats = feats.pooler_output
            elif hasattr(feats, 'last_hidden_state'): feats = feats.last_hidden_state[:, 0]
            all_logits.append(classifier(feats).cpu())
            all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


def compute_thresholds_with_temperature(logits, labels, T, target_precision=0.95):
    """Compute per-class thresholds on temperature-scaled probabilities."""
    probs = F.softmax(logits / T, dim=1)
    preds = probs.argmax(dim=1)
    confs = probs.max(dim=1).values

    print(f"\n  Computing thresholds (T={T:.4f}, target_precision={target_precision})")
    thresholds = {}

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        # Get all samples predicted as this class
        pred_mask = preds == cls_idx
        if pred_mask.sum() == 0:
            thresholds[cls_name.lower()] = 0.5
            continue

        pred_confs = confs[pred_mask].float()
        pred_labels = labels[pred_mask]
        pred_correct = (pred_labels == cls_idx)

        # Sort by confidence descending
        sorted_idx = torch.argsort(pred_confs, descending=True)
        pred_confs = pred_confs[sorted_idx]
        pred_correct = pred_correct[sorted_idx]

        # Find threshold where cumulative precision >= target
        cum_correct = torch.cumsum(pred_correct.float(), dim=0)
        cum_total = torch.arange(1, len(pred_correct) + 1, dtype=torch.float32)
        cum_precision = cum_correct / cum_total

        # Find the lowest confidence where precision is still >= target
        valid = cum_precision >= target_precision
        if valid.any():
            last_valid_idx = valid.nonzero()[-1].item()
            thresh = float(pred_confs[last_valid_idx])
        else:
            # Can't achieve target precision, use max confidence as threshold
            thresh = float(pred_confs[0]) + 0.01

        # Stats
        n_above = (pred_confs >= thresh).sum().item()
        if n_above > 0:
            actual_prec = float(pred_correct[pred_confs >= thresh].float().mean())
        else:
            actual_prec = 0.0

        conf_correct = float(pred_confs[pred_correct].mean()) if pred_correct.any() else 0
        conf_wrong = float(pred_confs[~pred_correct].mean()) if (~pred_correct).any() else 0

        thresholds[cls_name.lower()] = round(thresh, 4)

        print(f"    {cls_name:12s}: thresh={thresh:.4f}  "
              f"accept={n_above}/{pred_mask.sum()}  "
              f"precision={actual_prec:.3f}  "
              f"conf_correct={conf_correct:.3f}  "
              f"conf_wrong={conf_wrong:.3f}")

    return thresholds


def process_model(name, ckpt_path, config_path, load_vision_fn, get_normalize_fn):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    # Load config to get T
    with open(config_path) as f:
        cfg = json.load(f)
    T = cfg.get("temperature", 1.0)
    print(f"  Temperature: {T}")

    # Load model
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    vision_model = load_vision_fn(ckpt)
    classifier = ClassifierHead(ckpt["hidden_size"], NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])
    classifier = classifier.to(DEVICE).eval()
    normalize = get_normalize_fn(vision_model)
    transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

    loader = DataLoader(ValDataset(val_samples, transform, normalize),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    logits, labels_t = collect_logits(vision_model, classifier, loader)
    thresholds = compute_thresholds_with_temperature(logits, labels_t, T, TARGET_PRECISION)

    # Update config
    cfg["class_thresholds"] = thresholds
    with open(config_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"\n  ✅ Saved updated thresholds to {config_path}")
    print(f"     {thresholds}")

    del vision_model, classifier
    torch.cuda.empty_cache()
    return thresholds


# ── Collect samples ──
print("Collecting samples...")
all_samples = []
for root in TRAIN_ROOTS:
    if not os.path.isdir(root): continue
    for cls_name in sorted(os.listdir(root)):
        cls_path = os.path.join(root, cls_name)
        if not os.path.isdir(cls_path): continue
        label = map_class(cls_name)
        if label is None: continue
        for f in os.listdir(cls_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                all_samples.append((os.path.join(cls_path, f), label))

labels = [l for _, l in all_samples]
_, val_samples = train_test_split(all_samples, test_size=0.2, stratify=labels, random_state=42)
print(f"  Val: {len(val_samples)} samples")


# ── DINOv3 ──
def load_dino(ckpt):
    vm = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=False, num_classes=0)
    vm.load_state_dict(ckpt["vision_model"])
    return vm.to(DEVICE).eval()

def dino_normalize(vm):
    _cfg = timm.data.resolve_data_config(model=vm)
    return transforms.Normalize(mean=_cfg["mean"], std=_cfg["std"])

dino_thresh = process_model("DINOv3", DINOV3_CHECKPOINT, DINOV3_CONFIG, load_dino, dino_normalize)


# ── SigLIP2 ──
def load_siglip(ckpt):
    try:
        from transformers import Siglip2VisionModel
        vm = Siglip2VisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
    except:
        try:
            from transformers import SiglipVisionModel
            vm = SiglipVisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
        except:
            from transformers import AutoModel
            full = AutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
            vm = full.vision_model; del full
    vm.load_state_dict(ckpt["vision_model"])
    return vm.to(DEVICE).eval()

def siglip_normalize(vm):
    return transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

siglip_thresh = process_model("SigLIP2", SIGLIP2_CHECKPOINT, SIGLIP2_CONFIG, load_siglip, siglip_normalize)


print(f"\n{'='*60}")
print(f"  DONE — New thresholds (post temperature scaling)")
print(f"{'='*60}")
print(f"  DINOv3  (T=1.83): {dino_thresh}")
print(f"  SigLIP2 (T=2.67): {siglip_thresh}")
print(f"\n  Copy updated JSON configs to Orin and restart.")

  Val: 1934 samples

  DINOv3
  Temperature: 1.8306



  Computing thresholds (T=1.8306, target_precision=0.95)
    Upper       : thresh=0.6729  accept=1004/1054  precision=0.950  conf_correct=0.951  conf_wrong=0.779
    Lower       : thresh=0.6343  accept=707/748  precision=0.950  conf_correct=0.952  conf_wrong=0.732
    Underwear   : thresh=0.3535  accept=81/81  precision=0.988  conf_correct=0.949  conf_wrong=0.777
    Other       : thresh=0.5601  accept=50/51  precision=0.960  conf_correct=0.985  conf_wrong=0.649

  ✅ Saved updated thresholds to /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/dinov3_production_config.json
     {'upper': 0.6729, 'lower': 0.6343, 'underwear': 0.3535, 'other': 0.5601}

  SigLIP2
  Temperature: 2.6698


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

You are using a model of type siglip_vision_model to instantiate a model of type siglip2_vision_model. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Siglip2VisionModel LOAD REPORT from: google/siglip2-so400m-patch14-384
Key                                                          | Status     |                                                                                                 
-------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------
text_model.encoder.layers.{0...26}.layer_norm2.weight        | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.self_attn.v_proj.bias     | UNEXPECTED |                                                                                                 
text_model.encoder.layers.{0...26}.mlp.fc1.bias              | U

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip2-so400m-patch14-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...26}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.bias   


  Computing thresholds (T=2.6698, target_precision=0.95)
    Upper       : thresh=0.5269  accept=1016/1025  precision=0.951  conf_correct=0.958  conf_wrong=0.817
    Lower       : thresh=0.7480  accept=726/780  precision=0.950  conf_correct=0.956  conf_wrong=0.786
    Underwear   : thresh=0.5078  accept=79/79  precision=1.000  conf_correct=0.959  conf_wrong=0.000
    Other       : thresh=0.6313  accept=50/50  precision=0.960  conf_correct=0.968  conf_wrong=0.976

  ✅ Saved updated thresholds to /content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/siglip2_production_config.json
     {'upper': 0.5269, 'lower': 0.748, 'underwear': 0.5078, 'other': 0.6313}

  DONE — New thresholds (post temperature scaling)
  DINOv3  (T=1.83): {'upper': 0.6729, 'lower': 0.6343, 'underwear': 0.3535, 'other': 0.5601}
  SigLIP2 (T=2.67): {'upper': 0.5269, 'lower': 0.748, 'underwear': 0.5078, 'other': 0.6313}

  Copy updated JSON configs to Orin and restart.
